# Option C — Hybrid MoGe + LiDAR Pointmap for SAM3D

**Goal**: Fuse MoGe monocular depth with LiDAR metric measurements into a single dense metric pointmap **before** SAM3D inference. This is the paper's *intended* approach — the pose decoder receives metric-scale depth and can output translation already in meters, without any post-hoc rescaling.

## Contrast with Options A/B (post-hoc anchoring)

| Approach | When LiDAR is used | What SAM3D sees |
|---|---|---|
| Options A/B/v1-v3 | **After** inference (post-hoc scale/translate) | MoGe-only, non-metric pointmap |
| **Option C (this notebook)** | **Before** inference (fused into pointmap) | Metric pointmap (MoGe × affine fit to LiDAR) |

## Pipeline

1. Load nuScenes frame + LiDAR sweep (same as existing notebook)
2. Run MoGe to get raw (non-metric) pointmap
3. Fit global affine `Z_lidar = a × Z_moge + b` at in-image LiDAR pixels
4. Apply affine globally → dense metric depth map
5. Back-project to XYZ using nuScenes ground-truth intrinsics → fused metric pointmap in P3D space
6. Pass fused pointmap to `inference_model(rgb, mask, pointmap=...)` → SAM3D outputs metric translation directly
7. Compare Option C (raw) vs Option C + post-hoc correction vs MoGe-only + post-hoc

In [126]:
# ── Imports ───────────────────────────────────────────────────────────────────
import sys
import numpy as np
import cv2
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path

matplotlib.rcParams['figure.dpi'] = 120
%matplotlib inline

print('Imports OK')

In [127]:
# ── Configuration ─────────────────────────────────────────────────────────────
# Step 1: set FRAME_IDX and run through §3 to see all detected cars.
# Step 2: set OBJ_IDX to the index of the car you want to analyse, then re-run §4 onward.

DATA_ROOT          = '/media/lleba/ECP_Nuscenes_01/output2/ecp2nuscenes'
SAM3D_REPO         = Path('/home/lleba/Thesis/Development/SAM3D/sam-3d-objects')
CKPT_CONFIG        = SAM3D_REPO / 'checkpoints' / 'hf' / 'pipeline.yaml'

NUSCENES_SCENE_IDX = 10
FRAME_IDX          = 580   # ← change to test different frames
OBJ_IDX            = 1     # ← set AFTER viewing the all-masks overview in §3

SAM3_DIR           = Path('/media/lleba/ECP_Nuscenes_01/SAM3_Visualizations/Strassbourg')
SAM3_SCENE_PREFIX  = f'scene_{NUSCENES_SCENE_IDX}_cam_0_sam3_outputs__'

MIN_PROB     = 0.6
MIN_MASK_PX  = 100

# Minimum in-mask LiDAR hits to trust the depth anchor
MIN_LIDAR_HITS = 3

# Maximum plausible car depth span in meters (used for depth-window filtering)
MAX_CAR_DEPTH_SPAN = 4

# Robust MAD outlier filter (applied after depth-window filter, before local affine fit)
MAD_SIGMA_THRESH = 3.0   # reject points whose |residual| > MAD_SIGMA_THRESH * sigma_MAD
MIN_MOGE_SPAN   = 0.05   # min MoGe depth range (normalised) to attempt fit
MIN_LIDAR_SPAN  = 0.30   # min LiDAR depth span (m) to attempt fit

# Expected car dimensions [W, H, D] in meters (used in metrics)
CAR_PRIOR_DIMS = np.array([1.9, 1.5, 4.5])


print(f'Config: scene={NUSCENES_SCENE_IDX}  frame={FRAME_IDX}  obj={OBJ_IDX}')

## Setup: SAM3D model + helper functions

Load the SAM3D-Objects model once per session. Defines `mesh_to_r3`, `project`, `draw_mesh_on_image`, and `compute_metrics`.
All downstream sections depend on these.


In [140]:
# Load SAM3D-Objects model (run this cell once per notebook session)
import torch
sys.path.insert(0, str(SAM3D_REPO / 'notebook'))
from inference import Inference

inference_model = Inference(str(CKPT_CONFIG), compile=False)
print('SAM3D-Objects model ready.')

In [141]:
def mesh_to_r3(output):
    """object-local → R3 (OpenCV) camera space. Returns (verts (V,3), faces (F,3))."""
    from pytorch3d.renderer import look_at_view_transform
    from pytorch3d.transforms import Transform3d, quaternion_to_matrix
    from sam3d_objects.data.dataset.tdfy.transforms_3d import compose_transform

    def _sq(t):
        while t.dim() > 2: t = t.squeeze(0)
        return t

    quat  = _sq(output['rotation'].detach().cpu().float())
    trans = _sq(output['translation'].detach().cpu().float())
    scale = _sq(output['scale'].detach().cpu().float())
    if quat.dim() == 1:
        quat = quat.unsqueeze(0)

    l2c = compose_transform(
        scale=scale, rotation=quaternion_to_matrix(quat), translation=trans,
    )
    R_r3_to_p3d, _ = look_at_view_transform(
        eye=torch.tensor([[0., 0., -1.]]),
        at=torch.tensor([[0., 0.,  0.]]),
        up=torch.tensor([[0., -1., 0.]]),
    )
    t_p3d_to_r3 = Transform3d().rotate(R_r3_to_p3d).inverse()

    mesh_res    = output['mesh'][0]
    verts_local = mesh_res.vertices.float().cpu()
    faces       = mesh_res.faces.long().cpu().numpy().astype(np.int32)
    xyz_p3d     = l2c.transform_points(verts_local.unsqueeze(0)).squeeze(0)
    verts_r3    = (
        t_p3d_to_r3.transform_points(xyz_p3d.unsqueeze(0))
        .squeeze(0).numpy().astype(np.float32)
    )
    return verts_r3, faces


def project(pts, K):
    """(N,3) R3 → (N,2) pixel. Points behind camera → (-9999, -9999)."""
    valid = pts[:, 2] > 1e-6
    u = np.where(valid, K[0,0]*pts[:,0]/np.maximum(pts[:,2],1e-6)+K[0,2], -9999.)
    v = np.where(valid, K[1,1]*pts[:,1]/np.maximum(pts[:,2],1e-6)+K[1,2], -9999.)
    return np.stack([u, v], axis=1)


def draw_mesh_on_image(base_rgb, verts_r3, K, color=(255, 100, 0)):
    """Project mesh vertices onto image as colored dots."""
    out = base_rgb.copy()
    pts2d = project(verts_r3, K)
    u = np.round(pts2d[:, 0]).astype(int)
    v = np.round(pts2d[:, 1]).astype(int)
    valid = (u >= 0) & (u < out.shape[1]) & (v >= 0) & (v < out.shape[0])
    out[v[valid], u[valid]] = color
    return out


print('Helper functions defined: mesh_to_r3, project, draw_mesh_on_image')

## §1. Load nuScenes calibration + nearest LiDAR sweep

In [128]:
from nuscenes.nuscenes import NuScenes
from pyquaternion import Quaternion

print('Loading nuScenes...')
nusc  = NuScenes(version='v1.0-trainval', dataroot=DATA_ROOT, verbose=False)
scene = nusc.scene[NUSCENES_SCENE_IDX]
print(f'  Scene: {scene["name"]}')

# Walk CAM_FRONT frames
cam_token = nusc.get('sample', scene['first_sample_token'])['data']['CAM_FRONT']
frame_records = []
while cam_token:
    rec = nusc.get('sample_data', cam_token)
    frame_records.append(rec)
    cam_token = rec['next']
print(f'  {len(frame_records)} CAM_FRONT frames')

rec_cam  = frame_records[FRAME_IDX]
img_path = nusc.get_sample_data_path(rec_cam['token'])

cal_cam   = nusc.get('calibrated_sensor', rec_cam['calibrated_sensor_token'])
K_np      = np.array(cal_cam['camera_intrinsic'], dtype=np.float64)   # (3,3)
fx, fy    = K_np[0,0], K_np[1,1]
cx, cy    = K_np[0,2], K_np[1,2]
R_cam2ego = Quaternion(cal_cam['rotation']).rotation_matrix            # (3,3)
t_cam2ego = np.array(cal_cam['translation'], dtype=np.float64)         # (3,)
R_ego2cam = R_cam2ego.T
t_ego2cam = -R_ego2cam @ t_cam2ego

print(f'  ECP intrinsics: fx={fx:.1f}  fy={fy:.1f}  cx={cx:.1f}  cy={cy:.1f}')
print(f'  Image: {img_path}')

In [129]:
# Find LiDAR sweep nearest in time to this camera frame
lt = nusc.get('sample', scene['first_sample_token'])['data']['LIDAR_TOP']
lidar_records = []
while lt:
    lr = nusc.get('sample_data', lt)
    lidar_records.append(lr)
    lt = lr['next']

lidar_ts  = np.array([lr['timestamp'] for lr in lidar_records], dtype=np.int64)
best_idx  = int(np.argmin(np.abs(lidar_ts - rec_cam['timestamp'])))
rec_lidar = lidar_records[best_idx]
dt_ms     = abs(rec_lidar['timestamp'] - rec_cam['timestamp']) / 1000.0

lidar_path = nusc.get_sample_data_path(rec_lidar['token'])
cal_lidar  = nusc.get('calibrated_sensor', rec_lidar['calibrated_sensor_token'])
R_l2e      = Quaternion(cal_lidar['rotation']).rotation_matrix     # (3,3) lidar→ego
t_l2e      = np.array(cal_lidar['translation'], dtype=np.float64)  # (3,)

print(f'  LiDAR sweep: {lidar_path}')
print(f'  Time offset to camera: {dt_ms:.1f} ms')

## §2. Transform LiDAR → camera space & project to image

In [130]:
# Load LiDAR point cloud (.bin = float32, columns: x y z intensity ring)
pc_raw    = np.fromfile(lidar_path, dtype=np.float32).reshape(-1, 5)
pts_lidar = pc_raw[:, :3].T   # (3, N) in LiDAR frame
print(f'  Loaded {pts_lidar.shape[1]:,} LiDAR points')

# lidar → ego → camera (R3/OpenCV: X right, Y down, Z forward)
pts_ego = R_l2e @ pts_lidar + t_l2e[:, np.newaxis]
pts_cam = R_ego2cam @ pts_ego + t_ego2cam[:, np.newaxis]

# Keep only points in front of camera
in_front = pts_cam[2] > 0.5
pts_cam  = pts_cam[:, in_front]
print(f'  {pts_cam.shape[1]:,} points in front of camera')

# Pinhole projection
Z_all = pts_cam[2]
u_all = fx * pts_cam[0] / Z_all + cx
v_all = fy * pts_cam[1] / Z_all + cy

img_bgr = cv2.imread(img_path)
H, W    = img_bgr.shape[:2]
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

# Keep only points that project within image bounds
in_img      = (u_all >= 0) & (u_all < W) & (v_all >= 0) & (v_all < H)
u_vis       = u_all[in_img]
v_vis       = v_all[in_img]
Z_vis       = Z_all[in_img]
pts_cam_vis = pts_cam[:, in_img]
print(f'  {pts_cam_vis.shape[1]:,} points project into image')
xyz_lid = pts_lidar.T.astype(np.float64)  # (N,3) LiDAR frame


In [131]:
# Visualize all projected LiDAR points — colored by depth
fig, ax = plt.subplots(1, 1, figsize=(16, 8))
ax.imshow(img_rgb)
sc = ax.scatter(u_vis, v_vis, c=Z_vis, cmap='plasma_r', s=2, alpha=0.7, vmin=0, vmax=50)
plt.colorbar(sc, ax=ax, label='depth (m)')
ax.set_title(f'All LiDAR projected to image — frame {FRAME_IDX}')
ax.axis('off')
plt.tight_layout()
plt.show()

## §3. Load SAM3 masks — overview of ALL detected cars

Run this section to see every detected car with its index label.
Set `OBJ_IDX` in the config cell to the instance you want to analyse.

In [132]:
# Load SAM3 car masks for this frame
mask_path = SAM3_DIR / f'{SAM3_SCENE_PREFIX}car.npy'
sam3_data = np.load(str(mask_path), allow_pickle=True).item()

frame      = sam3_data[FRAME_IDX]
obj_ids    = frame['out_obj_ids']
probs      = np.asarray(frame['out_probs'],       dtype=np.float32)
raw_masks  = frame['out_binary_masks']
boxes_xywh = np.asarray(frame['out_boxes_xywh'],  dtype=np.float32)

# Collect all valid detections
detections = []
for i, (oid, prob) in enumerate(zip(obj_ids, probs)):
    if float(prob) < MIN_PROB:
        continue
    bm = np.asarray(raw_masks[i], dtype=bool)
    if bm.sum() < MIN_MASK_PX:
        continue
    bx = boxes_xywh[i]
    detections.append({
        'obj_id': int(oid), 'prob': float(prob),
        'binary_mask': bm,
        'box_xyxy': [float(bx[0])*W, float(bx[1])*H,
                     (float(bx[0])+float(bx[2]))*W,
                     (float(bx[1])+float(bx[3]))*H],
    })

print(f'{len(detections)} car detection(s) above threshold in frame {FRAME_IDX}:')
for i, d in enumerate(detections):
    n_px = d['binary_mask'].sum()
    b    = d['box_xyxy']
    print(f'  [{i}] obj_id={d["obj_id"]}  prob={d["prob"]:.3f}  '
          f'mask_px={n_px}  bbox=[{b[0]:.0f},{b[1]:.0f},{b[2]:.0f},{b[3]:.0f}]')

In [133]:
# ── Overview: all detected cars on a single image, each with index label ──────
_COLORS = [
    (0, 230, 0),    # green
    (255, 80, 0),   # orange
    (0, 180, 255),  # blue
    (255, 0, 200),  # magenta
    (255, 230, 0),  # yellow
    (0, 255, 200),  # teal
    (180, 0, 255),  # purple
    (255, 255, 255),# white
]

overview = img_rgb.copy().astype(np.float32)
for idx, det in enumerate(detections):
    color = _COLORS[idx % len(_COLORS)]
    mask  = det['binary_mask']
    fill  = np.zeros_like(overview)
    fill[mask] = color
    overview = overview * 0.65 + fill * 0.35

overview = overview.clip(0, 255).astype(np.uint8)

fig, ax = plt.subplots(1, 1, figsize=(16, 8))
ax.imshow(overview)

for idx, det in enumerate(detections):
    color_f = tuple(c/255 for c in _COLORS[idx % len(_COLORS)])
    x1, y1, x2, y2 = det['box_xyxy']
    rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                              lw=2, edgecolor=color_f, facecolor='none')
    ax.add_patch(rect)
    ax.text(x1+4, y1+20, f'[{idx}]',
            color=color_f, fontsize=14, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.2', fc='black', alpha=0.6))
    ax.text(x1+4, y1+42, f'p={det["prob"]:.2f}',
            color=color_f, fontsize=9,
            bbox=dict(boxstyle='round,pad=0.2', fc='black', alpha=0.5))

ax.set_title(
    f'All detected cars — frame {FRAME_IDX}  ({len(detections)} instances)\n'
    f'→ Set OBJ_IDX in the config cell to the index you want to analyse',
    fontsize=12
)
ax.axis('off')
plt.tight_layout()
plt.show()

In [134]:
# ── Select the one instance we will process ───────────────────────────────────
assert OBJ_IDX < len(detections), \
    f'OBJ_IDX={OBJ_IDX} out of range — only {len(detections)} detections'

det         = detections[OBJ_IDX]
binary_mask = det['binary_mask']   # (H, W) bool
x1, y1, x2, y2 = det['box_xyxy']
sel_color   = _COLORS[OBJ_IDX % len(_COLORS)]
sel_color_f = tuple(c/255 for c in sel_color)

overlay_sel = img_rgb.copy().astype(np.float32)
fill = np.zeros_like(overlay_sel)
fill[binary_mask] = sel_color
overlay_sel = (overlay_sel * 0.6 + fill * 0.4).clip(0, 255).astype(np.uint8)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].imshow(overlay_sel)
rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                          lw=3, edgecolor=sel_color_f, facecolor='none')
axes[0].add_patch(rect)
axes[0].text(x1+4, y1+22, f'[{OBJ_IDX}] selected',
             color=sel_color_f, fontsize=13, fontweight='bold',
             bbox=dict(boxstyle='round,pad=0.2', fc='black', alpha=0.65))
axes[0].set_title(f'Selected: obj {OBJ_IDX}  |  prob={det["prob"]:.3f}  |  '
                  f'mask_px={binary_mask.sum()}')
axes[0].axis('off')

pad = 40
axes[1].imshow(overlay_sel[
    max(0, int(y1)-pad):min(H, int(y2)+pad),
    max(0, int(x1)-pad):min(W, int(x2)+pad)
])
axes[1].set_title('Zoom — selected instance')
axes[1].axis('off')

plt.tight_layout()
plt.show()

print(f'\n→ Processing obj [{OBJ_IDX}]  '
      f'mask bbox: u=[{x1:.0f},{x2:.0f}]  v=[{y1:.0f},{y2:.0f}]  '
      f'size={x2-x1:.0f}×{y2-y1:.0f} px')

## §4. LiDAR hits inside the selected mask — find front-surface depth

In [135]:
# Snap projected LiDAR coords to nearest pixel and look up in mask
u_int = np.round(u_vis).astype(int)
v_int = np.round(v_vis).astype(int)

in_bounds = (u_int >= 0) & (u_int < W) & (v_int >= 0) & (v_int < H)
u_int = u_int[in_bounds]
v_int = v_int[in_bounds]
Z_v   = Z_vis[in_bounds]
u_f_v = u_vis[in_bounds]
v_f_v = v_vis[in_bounds]

in_mask = binary_mask[v_int, u_int]   # (M,) bool
n_hits  = int(in_mask.sum())
print(f'  {n_hits} LiDAR points project inside the selected car mask')

if n_hits < MIN_LIDAR_HITS:
    print(f'  WARNING: fewer than {MIN_LIDAR_HITS} hits → global affine fit may be unreliable')
    Z_lidar_anchor = None
else:
    Z_in_mask      = Z_v[in_mask]
    Z_lidar_anchor = float(Z_in_mask.min())   # nearest = front surface
    print(f'  Z_lidar_front  = {Z_lidar_anchor:.3f} m  (min, {n_hits} hits)')
    print(f'  Z_lidar_median = {float(np.median(Z_in_mask)):.3f} m')
    print(f'  Z_lidar_max    = {float(Z_in_mask.max()):.3f} m')

In [136]:
# Visualize: full frame + zoomed, in-mask hits highlighted
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, zoom in zip(axes, [False, True]):
    ax.imshow(img_rgb)

    contours, _ = cv2.findContours(
        binary_mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )
    for cnt in contours:
        cp = cnt[:, 0, :]
        ax.plot(np.append(cp[:,0], cp[0,0]),
                np.append(cp[:,1], cp[0,1]),
                color=sel_color_f, lw=1.5)

    if not zoom:
        ax.scatter(u_f_v[~in_mask], v_f_v[~in_mask],
                   c='white', s=2, alpha=0.25, label='LiDAR outside mask')

    if n_hits > 0:
        sc = ax.scatter(
            u_f_v[in_mask], v_f_v[in_mask],
            c=Z_v[in_mask], cmap='plasma_r',
            s=35, alpha=0.95, zorder=5, vmin=0, vmax=50,
            label=f'LiDAR in mask ({n_hits} pts)'
        )
        plt.colorbar(sc, ax=ax, label='depth (m)', fraction=0.025)

        ni   = int(np.argmin(Z_v[in_mask]))
        u_nr = u_f_v[in_mask][ni]
        v_nr = v_f_v[in_mask][ni]
        ax.scatter([u_nr], [v_nr], c='cyan', s=200, marker='*', zorder=10,
                   label=f'anchor  Z={Z_lidar_anchor:.2f} m')

    ax.legend(loc='upper right', fontsize=8)
    ax.axis('off')

    if zoom:
        pad = 70
        ax.set_xlim(max(0, x1-pad), min(W, x2+pad))
        ax.set_ylim(min(H, y2+pad), max(0, y1-pad))
        ax.set_title(f'Zoom — obj [{OBJ_IDX}]  (front Z={Z_lidar_anchor:.2f} m)' if Z_lidar_anchor else 'Zoom — fallback')
    else:
        ax.set_title(f'Full frame — LiDAR in-mask hits for obj [{OBJ_IDX}]')

plt.suptitle(f'Frame {FRAME_IDX} | obj [{OBJ_IDX}] | {n_hits} LiDAR hits in mask', fontsize=13)
plt.tight_layout()
plt.show()

## §4b. Depth-window filtering — extract valid object-surface LiDAR

Not all in-mask LiDAR points correspond to the object surface. Common contaminants:
- Background through car windows
- LiDAR reflections / multi-bounce returns
- Mask spillover (adjacent pixels belonging to background)
- Neighboring objects at different depths

**Algorithm**: sliding depth-window of width `MAX_CAR_DEPTH_SPAN` that maximises the count
of in-mask LiDAR points. This selects the densest depth cluster, which corresponds to the
actual vehicle surface.

In [137]:
# ── Depth-window filter ───────────────────────────────────────────────────────
# Expose in-mask arrays (needed here and in §6)
Z_in_mask_arr = Z_v[in_mask]       # metric depth of all in-mask hits
u_in_mask_f   = u_f_v[in_mask]     # float pixel u
v_in_mask_f   = v_f_v[in_mask]     # float pixel v

obj_filter_ok = False  # will be set True if filtering succeeds

if n_hits >= 2:
    Z_sorted = np.sort(Z_in_mask_arr)
    best_count, best_lo, best_hi = 0, Z_sorted[0], Z_sorted[0] + MAX_CAR_DEPTH_SPAN
    for z0 in Z_sorted:
        z1 = z0 + MAX_CAR_DEPTH_SPAN
        count = int(np.sum((Z_in_mask_arr >= z0) & (Z_in_mask_arr <= z1)))
        if count > best_count:
            best_count, best_lo, best_hi = count, float(z0), float(z1)

    valid_obj_mask = (Z_in_mask_arr >= best_lo) & (Z_in_mask_arr <= best_hi)
    n_valid_obj = int(valid_obj_mask.sum())
    n_rejected  = n_hits - n_valid_obj

    # Filtered arrays (used in §6 for local affine fit)
    u_obj = u_in_mask_f[valid_obj_mask]
    v_obj = v_in_mask_f[valid_obj_mask]
    Z_obj = Z_in_mask_arr[valid_obj_mask]
    obj_filter_ok = (n_valid_obj >= 3)

    print(f'Depth-window filter  span={MAX_CAR_DEPTH_SPAN} m:')
    print(f'  Window selected  : [{best_lo:.2f}, {best_hi:.2f}] m')
    print(f'  Valid object pts : {n_valid_obj}  ({100*n_valid_obj/max(1,n_hits):.0f}%)')
    print(f'  Rejected outliers: {n_rejected}  ({100*n_rejected/max(1,n_hits):.0f}%)')
    if obj_filter_ok:
        print(f'  Z_front (filtered): {Z_obj.min():.3f} m')
        print(f'  Z_median (filtered): {np.median(Z_obj):.3f} m')
else:
    valid_obj_mask = np.ones(n_hits, dtype=bool) if n_hits > 0 else np.array([], dtype=bool)
    u_obj = u_in_mask_f.copy(); v_obj = v_in_mask_f.copy(); Z_obj = Z_in_mask_arr.copy()
    n_valid_obj = n_hits; n_rejected = 0
    best_lo = float(Z_in_mask_arr.min()) if n_hits > 0 else 0.
    best_hi = best_lo + MAX_CAR_DEPTH_SPAN
    print(f'  Fewer than 2 in-mask hits — skipping depth-window filter')

In [138]:
# ── Viz: depth distribution + selected window ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if n_hits > 0:
    ax = axes[0]
    bins = np.linspace(Z_in_mask_arr.min() - 0.5, Z_in_mask_arr.max() + 0.5,
                       max(20, n_hits // 2 + 1))
    ax.hist(Z_in_mask_arr[~valid_obj_mask], bins=bins,
            color='tomato', alpha=0.8, label=f'rejected ({n_rejected})', zorder=3)
    ax.hist(Z_in_mask_arr[valid_obj_mask],  bins=bins,
            color='steelblue', alpha=0.85, label=f'valid obj ({n_valid_obj})', zorder=4)
    ax.axvspan(best_lo, best_hi, alpha=0.12, color='green', label=f'window [{best_lo:.1f}, {best_hi:.1f}] m')
    ax.axvline(best_lo, color='green', lw=2, linestyle='--')
    ax.axvline(best_hi, color='green', lw=2, linestyle='--')
    ax.set_xlabel('LiDAR depth (m)'); ax.set_ylabel('count')
    ax.set_title(f'In-mask LiDAR depth distribution\n'
                 f'obj [{OBJ_IDX}]  frame {FRAME_IDX}  |  total={n_hits}')
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

    ax2 = axes[1]
    if n_hits > 1:
        ax2.scatter(np.arange(len(Z_in_mask_arr)),
                    np.sort(Z_in_mask_arr),
                    c=['steelblue' if valid_obj_mask[np.argsort(Z_in_mask_arr)[i]] else 'tomato'
                       for i in range(n_hits)],
                    s=30, alpha=0.8)
        ax2.axhline(best_lo, color='green', lw=1.5, linestyle='--', label=f'lo={best_lo:.2f} m')
        ax2.axhline(best_hi, color='green', lw=1.5, linestyle='--', label=f'hi={best_hi:.2f} m')
        ax2.set_xlabel('rank (sorted by depth)'); ax2.set_ylabel('Z (m)')
        ax2.set_title('Sorted depths: valid (blue) vs rejected (red)')
        ax2.legend(fontsize=9); ax2.grid(alpha=0.3)
else:
    for ax in axes:
        ax.text(0.5, 0.5, 'No in-mask LiDAR hits', ha='center', va='center',
                transform=ax.transAxes, fontsize=14)

plt.suptitle(f'Depth-window filtering — obj [{OBJ_IDX}]', fontsize=12)
plt.tight_layout(); plt.show()

In [139]:
# ── Viz: filtered vs rejected LiDAR on image ──────────────────────────────────
ys_m, xs_m = np.where(binary_mask)
pad = 80
xl = max(0, int(xs_m.min()) - pad); xr = min(W, int(xs_m.max()) + pad)
yt = max(0, int(ys_m.min()) - pad); yb = min(H, int(ys_m.max()) + pad)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
for ax, do_zoom in zip(axes, [False, True]):
    ax.imshow(img_rgb)
    # Mask outline
    contours, _ = cv2.findContours(binary_mask.astype(np.uint8),
                                    cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for cnt in contours:
        cp = cnt[:, 0, :]
        ax.plot(np.append(cp[:,0], cp[0,0]), np.append(cp[:,1], cp[0,1]),
                color=sel_color_f, lw=1.5)

    if n_hits > 0:
        # Rejected (red)
        if n_rejected > 0:
            ax.scatter(u_in_mask_f[~valid_obj_mask], v_in_mask_f[~valid_obj_mask],
                       c='red', s=50, alpha=0.85, zorder=5, label=f'rejected ({n_rejected})',
                       edgecolors='darkred', linewidths=0.5)
        # Valid (colored by depth)
        if n_valid_obj > 0:
            sc = ax.scatter(u_obj, v_obj, c=Z_obj, cmap='plasma_r', s=60, alpha=0.95,
                            zorder=6, vmin=best_lo - 1, vmax=best_hi + 1,
                            label=f'valid obj ({n_valid_obj})',
                            edgecolors='white', linewidths=0.3)
            plt.colorbar(sc, ax=ax, label='depth (m)', fraction=0.025)

    ax.legend(loc='upper right', fontsize=9)
    ax.axis('off')
    if do_zoom:
        ax.set_xlim(xl, xr); ax.set_ylim(yb, yt)
        ax.set_title(f'Zoom — filtered in-mask LiDAR  obj [{OBJ_IDX}]')
    else:
        ax.set_title(f'Full frame — depth-window filtered LiDAR  frame {FRAME_IDX}')

plt.suptitle(f'Frame {FRAME_IDX} | obj [{OBJ_IDX}] | valid={n_valid_obj}  rejected={n_rejected}',
             fontsize=12)
plt.tight_layout(); plt.show()

## §5. Load SAM3D model + helper functions

Load the model once. Define `mesh_to_r3`, `project`, and `draw_mesh_on_image` here so they are available for all later sections.

## §6. Build fused metric pointmaps — Global vs Local calibration

Two parallel calibration strategies:

| Strategy | Fit data | Scope |
|----------|----------|-------|
| **Global** | All valid in-image LiDAR hits (full scene) | Good when scene has varied depths |
| **Local** | Depth-window-filtered in-mask cluster only | Better for per-object accuracy |

Both use the **verified** R3→P3D coordinate conversion:
`X_p3d = -X_r3, Y_p3d = -Y_r3, Z_p3d = Z_r3`
(R = diag(-1,-1,+1), verified numerically)

### §6a. Get MoGe raw pointmap

In [142]:
rgb = img_bgr[:, :, ::-1].copy()
rgba_full = np.concatenate([rgb, np.full((H, W, 1), 255, dtype=np.uint8)], axis=-1)

with inference_model._pipeline.device:
    ptmap_dict = inference_model._pipeline.compute_pointmap(rgba_full)

norm_K  = ptmap_dict['intrinsics'].cpu().numpy()
K_moge  = np.array([[norm_K[0,0]*W, 0., norm_K[0,2]*W],
                     [0., norm_K[1,1]*H, norm_K[1,2]*H],
                     [0., 0., 1.]], dtype=np.float64)
# moge_ptmap_p3d: (H,W,3) in P3D space — Z channel == R3 depth (identical under the transform)
moge_ptmap_p3d = ptmap_dict['pointmap'].cpu().permute(1, 2, 0)
Z_moge_map     = moge_ptmap_p3d[:, :, 2].numpy()   # (H,W) non-metric depth

print(f'MoGe K: fx={K_moge[0,0]:.1f}  fy={K_moge[1,1]:.1f}  cx={K_moge[0,2]:.1f}  cy={K_moge[1,2]:.1f}')
print(f'GT   K: fx={fx:.1f}  fy={fy:.1f}  cx={cx:.1f}  cy={cy:.1f}')
print(f'Z_moge range: [{np.nanmin(Z_moge_map):.4f}, {np.nanmax(Z_moge_map):.4f}]')

In [143]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
im = axes[0].imshow(Z_moge_map, cmap='plasma_r')
plt.colorbar(im, ax=axes[0], label='Z_moge (non-metric)')
axes[0].set_title('MoGe raw depth map (non-metric units)')
axes[0].axis('off')

# Zoom to mask region
axes[1].imshow(Z_moge_map[yt:yb, xl:xr], cmap='plasma_r')
axes[1].set_title(f'Zoom to obj [{OBJ_IDX}] region\n'
                  f'MoGe Z in mask: [{np.nanmin(Z_moge_map[yt:yb,xl:xr]):.3f}, {np.nanmax(Z_moge_map[yt:yb,xl:xr]):.3f}]')
axes[1].axis('off')
plt.suptitle('MoGe raw pointmap depth', fontsize=12)
plt.tight_layout(); plt.show()

### §6b. Sample MoGe depth at LiDAR locations & fit both affines

In [144]:
# ── Global: sample Z_moge at ALL in-image LiDAR pixel locations ───────────────
Z_moge_all = Z_moge_map[v_int, u_int]   # (N_in_image,)
valid_global = np.isfinite(Z_moge_all) & (Z_v > 0.5) & (Z_v < 80.0)
Z_moge_g = Z_moge_all[valid_global]
Z_lidar_g = Z_v[valid_global]
n_fit_g   = int(valid_global.sum())

A_g = np.stack([Z_moge_g, np.ones_like(Z_moge_g)], axis=1)
(a_global, b_global), _, _, _ = np.linalg.lstsq(A_g, Z_lidar_g, rcond=None)

Z_pred_g = a_global * Z_moge_g + b_global
ss_res_g  = float(np.sum((Z_lidar_g - Z_pred_g)**2))
ss_tot_g  = float(np.sum((Z_lidar_g - Z_lidar_g.mean())**2))
r2_g      = 1.0 - ss_res_g / ss_tot_g
rmse_g    = float(np.sqrt(ss_res_g / n_fit_g))
rel_err_g = (Z_pred_g - Z_lidar_g) / Z_lidar_g

print(f'Global fit  (n={n_fit_g:,}):  Z = {a_global:.4f} × Z_moge + {b_global:.4f}')
print(f'  R²={r2_g:.4f}  RMSE={rmse_g:.3f} m  median|rel_err|={np.median(np.abs(rel_err_g))*100:.1f}%')

In [145]:
# ── Local: sample Z_moge at depth-window-filtered in-mask pixels ─────────────
u_obj_int = np.round(u_obj).astype(int)
v_obj_int = np.round(v_obj).astype(int)
Z_moge_obj_samples = Z_moge_map[v_obj_int, u_obj_int]

valid_local_init = np.isfinite(Z_moge_obj_samples) & (Z_obj > 0.5)
Z_moge_l_init  = Z_moge_obj_samples[valid_local_init]
Z_lidar_l_init = Z_obj[valid_local_init]
n_fit_l_init   = int(valid_local_init.sum())

# ── Stage 2: robust MAD outlier rejection ─────────────────────────────────────
# After depth-window filtering some points may still be outliers in the
# MoGe→LiDAR affine relationship (glass reflections, grazing incidence, etc.).
# Strategy: fit initial affine, compute residuals, reject >MAD_SIGMA_THRESH σ_MAD,
# refit on inliers.  Multiple fallbacks if data are too sparse/flat.

local_fit_ok   = False
n_mad_rejected = 0
inlier_mask_l  = np.ones(max(n_fit_l_init, 0), dtype=bool)
a_local_init   = b_local_init = float('nan')
Z_moge_l       = Z_moge_l_init.copy()   # default: all depth-window points
Z_lidar_l      = Z_lidar_l_init.copy()
n_fit_l        = n_fit_l_init

if n_fit_l_init >= 3:
    moge_span  = float(Z_moge_l_init.max()  - Z_moge_l_init.min())
    lidar_span = float(Z_lidar_l_init.max() - Z_lidar_l_init.min())

    # ── Variation guard: if data are too flat, skip robust step ───────────────
    if moge_span < MIN_MOGE_SPAN or lidar_span < MIN_LIDAR_SPAN:
        print(f'[local fit] insufficient variation — '
              f'moge_span={moge_span:.4f}, lidar_span={lidar_span:.3f} m')
        print('  → using plain depth-window local fit (no MAD step)')
        A_l = np.stack([Z_moge_l, np.ones_like(Z_moge_l)], axis=1)
        (a_local, b_local), _, _, _ = np.linalg.lstsq(A_l, Z_lidar_l, rcond=None)
        a_local_init, b_local_init = a_local, b_local
        local_fit_ok = True

    else:
        # ── Stage 1: initial affine fit ───────────────────────────────────────
        A_l0 = np.stack([Z_moge_l_init, np.ones_like(Z_moge_l_init)], axis=1)
        (a_local_init, b_local_init), _, _, _ = np.linalg.lstsq(
            A_l0, Z_lidar_l_init, rcond=None)

        # ── Stage 2: MAD-based residual rejection ─────────────────────────────
        residuals_init = Z_lidar_l_init - (a_local_init * Z_moge_l_init + b_local_init)
        med_res        = float(np.median(residuals_init))
        mad            = float(np.median(np.abs(residuals_init - med_res)))
        sigma_mad      = 1.4826 * mad          # consistent scale estimate
        mad_thresh     = MAD_SIGMA_THRESH * sigma_mad

        inlier_mask_l  = np.abs(residuals_init - med_res) <= mad_thresh
        n_mad_rejected = int((~inlier_mask_l).sum())
        n_inliers      = int(inlier_mask_l.sum())

        print(f'[local fit stage 1]  n={n_fit_l_init}  '
              f'a={a_local_init:.4f}  b={b_local_init:.4f}')
        print(f'  MAD={mad:.4f}  σ_MAD={sigma_mad:.4f}  '
              f'thresh={mad_thresh:.4f} m  rejected={n_mad_rejected}')

        # ── Inlier guard ──────────────────────────────────────────────────────
        if n_inliers < 3:
            print(f'  too few inliers ({n_inliers}) after MAD filter '
                  f'→ falling back to plain depth-window fit')
            inlier_mask_l  = np.ones(n_fit_l_init, dtype=bool)
            n_mad_rejected = 0
        else:
            Z_moge_l  = Z_moge_l_init[inlier_mask_l]
            Z_lidar_l = Z_lidar_l_init[inlier_mask_l]
            n_fit_l   = n_inliers

        # ── Stage 2 (final) robust refit ──────────────────────────────────────
        A_l = np.stack([Z_moge_l, np.ones_like(Z_moge_l)], axis=1)
        (a_local, b_local), _, _, _ = np.linalg.lstsq(A_l, Z_lidar_l, rcond=None)
        local_fit_ok = True

else:
    # Not enough points
    n_mad_rejected = 0
    print(f'[local fit] n_fit_l_init={n_fit_l_init} < 3 — skipping, will use global')

# ── Final fallback: if local fit unavailable, use global ──────────────────────
if not local_fit_ok:
    a_local, b_local = a_global, b_global
    r2_l = rmse_l = float('nan')
    rel_err_l = np.array([])
    print(f'Local fit FAILED → using global fallback '
          f'(a={a_global:.4f}, b={b_global:.4f})')
else:
    Z_pred_l  = a_local * Z_moge_l + b_local
    ss_res_l  = float(np.sum((Z_lidar_l - Z_pred_l) ** 2))
    ss_tot_l  = float(np.sum((Z_lidar_l - Z_lidar_l.mean()) ** 2)) if len(Z_lidar_l) > 1 else 1.
    r2_l      = 1.0 - ss_res_l / ss_tot_l if ss_tot_l > 0 else float('nan')
    rmse_l    = float(np.sqrt(ss_res_l / n_fit_l))
    rel_err_l = (Z_pred_l - Z_lidar_l) / np.maximum(Z_lidar_l, 1e-6)

    print(f'\n[local fit final]  n_inliers={n_fit_l}  '
          f'Z = {a_local:.4f} × Z_moge + {b_local:.4f}')
    print(f'  R²={r2_l:.4f}  RMSE={rmse_l:.3f} m  '
          f'median|rel_err|={np.median(np.abs(rel_err_l))*100:.1f}%')

print(f'\nScale comparison:  a_global={a_global:.4f}  a_local={a_local:.4f}  '
      f'ratio={a_local/a_global:.3f}')
print(f'Offset comparison: b_global={b_global:.4f}  b_local={b_local:.4f}')


In [146]:
# ── Viz: robust MAD outlier rejection — scatter + residuals ──────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── Panel 1: MoGe vs LiDAR scatter with both fit lines ───────────────────────
ax = axes[0]
if n_fit_l_init > 0:
    out_mask = ~inlier_mask_l
    z_lo = Z_moge_l_init.min() - 0.02
    z_hi = Z_moge_l_init.max() + 0.02
    z_rng = np.linspace(z_lo, z_hi, 200)

    # Outliers (red)
    if n_mad_rejected > 0:
        ax.scatter(Z_moge_l_init[out_mask], Z_lidar_l_init[out_mask],
                   c='tomato', s=70, alpha=0.9, zorder=4,
                   label=f'MAD outliers ({n_mad_rejected})',
                   edgecolors='darkred', linewidths=0.6)

    # Inliers (green)
    ax.scatter(Z_moge_l_init[inlier_mask_l], Z_lidar_l_init[inlier_mask_l],
               c='mediumseagreen', s=70, alpha=0.85, zorder=5,
               label=f'inliers ({int(inlier_mask_l.sum())})',
               edgecolors='darkgreen', linewidths=0.6)

    # Initial fit (dashed orange)
    if np.isfinite(a_local_init):
        ax.plot(z_rng, a_local_init * z_rng + b_local_init,
                color='darkorange', lw=1.8, linestyle='--', zorder=3,
                label=f'initial: {a_local_init:.3f}×Z + {b_local_init:.3f}')

    # Robust fit (solid crimson)
    if local_fit_ok:
        ax.plot(z_rng, a_local * z_rng + b_local,
                color='crimson', lw=2.5, linestyle='-', zorder=6,
                label=f'robust: {a_local:.3f}×Z + {b_local:.3f}')

    ax.set_xlabel('Z_moge (normalised)')
    ax.set_ylabel('Z_lidar (m)')
    ax.set_title(f'MoGe vs LiDAR — robust MAD filter\n'
                 f'obj [{OBJ_IDX}]  frame {FRAME_IDX}  '
                 f'| σ-thresh={MAD_SIGMA_THRESH}')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
else:
    ax.text(0.5, 0.5, 'No points for local fit',
            ha='center', va='center', transform=ax.transAxes, fontsize=13)

# ── Panel 2: initial-fit residuals vs point index with MAD bands ──────────────
ax2 = axes[1]
if n_fit_l_init > 0 and np.isfinite(a_local_init):
    residuals_plot = Z_lidar_l_init - (a_local_init * Z_moge_l_init + b_local_init)
    xs = np.arange(n_fit_l_init)
    out_mask = ~inlier_mask_l

    if n_mad_rejected > 0:
        ax2.scatter(xs[out_mask], residuals_plot[out_mask],
                    c='tomato', s=70, alpha=0.9, zorder=4,
                    label=f'rejected ({n_mad_rejected})',
                    edgecolors='darkred', linewidths=0.6)
    ax2.scatter(xs[inlier_mask_l], residuals_plot[inlier_mask_l],
                c='mediumseagreen', s=70, alpha=0.85, zorder=5,
                label=f'inliers ({int(inlier_mask_l.sum())})',
                edgecolors='darkgreen', linewidths=0.6)

    # MAD threshold bands
    med_r  = float(np.median(residuals_plot))
    mad_r  = float(np.median(np.abs(residuals_plot - med_r)))
    thr_r  = MAD_SIGMA_THRESH * 1.4826 * mad_r
    ax2.axhline(med_r,         color='black',  lw=1.0, linestyle=':',  label=f'median={med_r:.3f} m')
    ax2.axhline(med_r + thr_r, color='orange', lw=1.5, linestyle='--', label=f'±{MAD_SIGMA_THRESH}σ_MAD ({thr_r:.3f} m)')
    ax2.axhline(med_r - thr_r, color='orange', lw=1.5, linestyle='--')
    ax2.axhline(0,             color='black',  lw=0.8)

    ax2.set_xlabel('point index (depth-window filtered)')
    ax2.set_ylabel('residual  Z_lidar − (a·Z_moge + b)  (m)')
    ax2.set_title(f'Initial-fit residuals  (n={n_fit_l_init})\n'
                  f'MAD={mad_r:.4f}  σ_MAD={1.4826*mad_r:.4f}  thresh={thr_r:.4f} m')
    ax2.legend(fontsize=8); ax2.grid(alpha=0.3)
else:
    ax2.text(0.5, 0.5, 'MAD filter not applied\n(insufficient points or flat data)',
             ha='center', va='center', transform=ax2.transAxes, fontsize=11)

plt.suptitle(f'Local robust MAD outlier rejection — obj [{OBJ_IDX}]', fontsize=12)
plt.tight_layout(); plt.show()


In [147]:
fig, axes = plt.subplots(1, 3, figsize=(19, 6))

# Panel 1: global fit scatter
ax = axes[0]
ax.scatter(Z_moge_g, Z_lidar_g, c='steelblue', s=4, alpha=0.35, label='all in-image')
z_rng = np.linspace(Z_moge_g.min(), Z_moge_g.max(), 200)
ax.plot(z_rng, a_global * z_rng + b_global, 'r-', lw=2,
        label=f'{a_global:.3f}×Z + {b_global:.3f}')
ax.set_xlabel('Z_moge'); ax.set_ylabel('Z_lidar (m)')
ax.set_title(f'Global fit  R²={r2_g:.3f}  RMSE={rmse_g:.3f} m\n(n={n_fit_g:,} in-image pts)')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# Panel 2: local fit scatter
ax2 = axes[1]
if local_fit_ok:
    ax2.scatter(Z_moge_l, Z_lidar_l, c='mediumseagreen', s=20, alpha=0.75,
                label=f'robust inliers ({n_fit_l})')
    z_rng_l = np.linspace(max(Z_moge_l.min() - 0.05, 0), Z_moge_l.max() + 0.05, 100)
    ax2.plot(z_rng_l, a_local * z_rng_l + b_local, 'r-', lw=2,
             label=f'{a_local:.3f}×Z + {b_local:.3f}')
    ax2.set_title(f'Robust local fit  R²={r2_l:.3f}  RMSE={rmse_l:.3f} m\n'
                 f'(inliers={n_fit_l}  MAD-rejected={n_mad_rejected})')
else:
    ax2.text(0.5, 0.5, f'Robust local fit unavailable\n(n_init={n_fit_l_init} pts  inliers={n_fit_l})',
             ha='center', va='center', transform=ax2.transAxes, fontsize=12)
    ax2.set_title('Local fit (unavailable)')
ax2.set_xlabel('Z_moge'); ax2.set_ylabel('Z_lidar (m)')
ax2.legend(fontsize=8); ax2.grid(alpha=0.3)

# Panel 3: residual histograms
ax3 = axes[2]
ax3.hist(rel_err_g, bins=60, range=(-0.6, 0.6), color='steelblue', alpha=0.6,
         label=f'global  med={np.median(rel_err_g)*100:.1f}%', edgecolor='none')
if local_fit_ok and len(rel_err_l) > 0:
    ax3.hist(rel_err_l, bins=20, range=(-0.6, 0.6), color='darkorange', alpha=0.7,
             label=f'local   med={np.median(rel_err_l)*100:.1f}%', edgecolor='none')
ax3.axvline(0, color='black', lw=1)
ax3.set_xlabel('relative error'); ax3.set_ylabel('count')
ax3.set_title('Residual distributions'); ax3.legend(fontsize=8); ax3.grid(alpha=0.3)

plt.suptitle('Affine fits: Global vs Local', fontsize=12)
plt.tight_layout(); plt.show()

### §6c. Build both fused metric pointmaps

In [148]:
u_grid, v_grid = np.meshgrid(np.arange(W), np.arange(H))

def build_fused_ptmap(a, b, label):
    """Build (H,W,3) float32 fused P3D pointmap from affine params."""
    Z_m = a * Z_moge_map + b
    Z_m = np.where(np.isfinite(Z_moge_map), np.maximum(Z_m, 0.1), np.nan)
    X_r3 = (u_grid - cx) * Z_m / fx
    Y_r3 = (v_grid - cy) * Z_m / fy
    # R3 → P3D: both X and Y negated (R = diag(-1,-1,+1))
    ptmap = np.stack([-X_r3, -Y_r3, Z_m], axis=-1).astype(np.float32)
    n_nan = int(np.isnan(ptmap[:,:,2]).sum())
    print(f'{label}: a={a:.4f} b={b:.4f}  Z∈[{np.nanmin(Z_m):.1f},{np.nanmax(Z_m):.1f}] m  '
          f'NaN={n_nan:,}px')
    return ptmap, Z_m

ptmap_global_np, Z_metric_global = build_fused_ptmap(a_global, b_global, 'Global ptmap')
ptmap_local_np,  Z_metric_local  = build_fused_ptmap(a_local,  b_local,  'Local  ptmap')

In [149]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
vmax_z = min(np.nanpercentile(Z_metric_global, 95), 60.)

im0 = axes[0].imshow(Z_moge_map, cmap='plasma_r')
plt.colorbar(im0, ax=axes[0], label='MoGe units')
axes[0].set_title('MoGe raw depth (non-metric)')
axes[0].axis('off')

im1 = axes[1].imshow(Z_metric_global, cmap='plasma_r', vmin=0, vmax=vmax_z)
plt.colorbar(im1, ax=axes[1], label='meters')
axes[1].set_title(f'Global metric depth  (a={a_global:.3f})')
axes[1].axis('off')

im2 = axes[2].imshow(Z_metric_local, cmap='plasma_r', vmin=0, vmax=vmax_z)
plt.colorbar(im2, ax=axes[2], label='meters')
axes[2].set_title(f'Local metric depth  (a={a_local:.3f})')
axes[2].axis('off')

plt.suptitle('Depth maps: MoGe | Global | Local', fontsize=12)
plt.tight_layout(); plt.show()

# Zoom to object region
fig2, axes2 = plt.subplots(1, 3, figsize=(18, 5))
for ax, (data, title) in zip(axes2, [
    (Z_moge_map,     'MoGe (non-metric)'),
    (Z_metric_global, f'Global (a={a_global:.3f})'),
    (Z_metric_local,  f'Local  (a={a_local:.3f})'),
]):
    crop = data[yt:yb, xl:xr]
    im = ax.imshow(crop, cmap='plasma_r')
    plt.colorbar(im, ax=ax, fraction=0.04)
    ax.set_title(f'{title}\nZ∈[{np.nanmin(crop):.2f},{np.nanmax(crop):.2f}]')
    ax.axis('off')

plt.suptitle(f'Object region depth maps — obj [{OBJ_IDX}]', fontsize=12)
plt.tight_layout(); plt.show()

### §6d. Validate fused pointmaps against in-mask LiDAR

In [150]:
u_obj_int_v = np.round(u_obj).astype(int)
v_obj_int_v = np.round(v_obj).astype(int)

Z_fused_g_obj = Z_metric_global[v_obj_int_v, u_obj_int_v]
Z_fused_l_obj = Z_metric_local[v_obj_int_v,  u_obj_int_v]

print('Fused pointmap validation (filtered obj LiDAR vs fused depth at same pixels):')
for label, Z_fused in [('Global', Z_fused_g_obj), ('Local', Z_fused_l_obj)]:
    err = Z_fused - Z_obj
    print(f'  {label}: RMSE={np.sqrt((err**2).mean()):.3f} m  '
          f'mean_err={err.mean():+.3f} m  max|err|={np.abs(err).max():.3f} m')

In [151]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
lims = [min(Z_obj.min(), Z_fused_g_obj.min(), Z_fused_l_obj.min()) - 0.5,
        max(Z_obj.max(), Z_fused_g_obj.max(), Z_fused_l_obj.max()) + 0.5]

for ax, (label, Z_f, color) in zip(axes, [
    ('Global', Z_fused_g_obj, 'steelblue'),
    ('Local',  Z_fused_l_obj, 'darkorange'),
]):
    err = Z_f - Z_obj
    ax.scatter(Z_obj, Z_f, c=color, s=40, alpha=0.75, edgecolors='none', zorder=5)
    ax.plot(lims, lims, 'r--', lw=1.5, label='y=x (perfect)')
    ax.set_xlabel('Z_obj_LiDAR (m)  — filtered ground truth')
    ax.set_ylabel(f'Z_fused_{label.lower()} at same px (m)')
    ax.set_title(f'{label} validation\nRMSE={np.sqrt((err**2).mean()):.3f} m  '
                 f'mean_err={err.mean():+.3f} m')
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.suptitle(f'Fused pointmap vs filtered object LiDAR — obj [{OBJ_IDX}]', fontsize=12)
plt.tight_layout(); plt.show()

## §7. SAM3D inference — Global & Local pointmaps

Run SAM3D twice: once with the globally calibrated pointmap and once with the locally calibrated one.
Both use seed=42 for reproducibility.

In [152]:
from scipy.spatial import cKDTree  # needed for hybrid anchor later

def run_inference(ptmap_np, label):
    """Run SAM3D with a fused pointmap. Returns output dict."""
    ptmap_t = torch.tensor(ptmap_np, dtype=torch.float32)   # NaN kept for invalid_xyz_token
    print(f'Running SAM3D [{label}]...')
    out = inference_model(rgb, binary_mask, seed=42, pointmap=ptmap_t)
    print(f'  Done.  translation={out["translation"].squeeze().tolist()}')
    return out

output_global = run_inference(ptmap_global_np, 'Global')
output_local  = run_inference(ptmap_local_np,  'Local')

## §8. Extract meshes + objective metrics

Compute metrics for each raw mesh (no post-hoc correction yet):
- **Bbox IoU**: 2D projected bounding-box intersection-over-union vs SAM3 mask bbox
- **Mask coverage**: fraction of visible mesh vertices that project inside the mask
- **Z_front error**: predicted near-surface depth − LiDAR Z_front (from filtered cluster)
- **Dim error**: mean absolute deviation from car prior [1.9, 1.5, 4.5] m

In [153]:
# ── Metrics helper ──────────────────────────────────────────────────────────
def compute_metrics(verts_r3, binary_mask, K, Z_lidar_front, label=''):
    Hm, Wm = binary_mask.shape
    fx_k = float(K[0,0]); fy_k = float(K[1,1])
    cx_k = float(K[0,2]); cy_k = float(K[1,2])

    # Project all vertices
    valid = verts_r3[:,2] > 1e-6
    u_p = np.where(valid, fx_k*verts_r3[:,0]/np.maximum(verts_r3[:,2],1e-6)+cx_k, -9999.)
    v_p = np.where(valid, fy_k*verts_r3[:,1]/np.maximum(verts_r3[:,2],1e-6)+cy_k, -9999.)

    valid_proj = valid & (u_p > -9000)
    v2d = np.stack([u_p[valid_proj], v_p[valid_proj]], axis=1)

    if len(v2d) == 0:
        return {'label':label,'iou':0.,'coverage':0.,'Z_front':float('nan'),
                'Z_front_err':float('nan'),'dims':np.zeros(3),'ctr':np.zeros(3),
                'ctr_ego':np.zeros(3),'dim_err':float('nan'),'n_in_mask':0}

    bx1,by1 = float(v2d[:,0].min()), float(v2d[:,1].min())
    bx2,by2 = float(v2d[:,0].max()), float(v2d[:,1].max())

    ys_m2, xs_m2 = np.where(binary_mask)
    mx1,my1 = float(xs_m2.min()), float(ys_m2.min())
    mx2,my2 = float(xs_m2.max()), float(ys_m2.max())

    ix1,iy1 = max(bx1,mx1), max(by1,my1)
    ix2,iy2 = min(bx2,mx2), min(by2,my2)
    inter = max(0.,ix2-ix1)*max(0.,iy2-iy1)
    a_mesh = (bx2-bx1)*(by2-by1); a_mask = (mx2-mx1)*(my2-my1)
    iou = inter/(a_mesh+a_mask-inter) if (a_mesh+a_mask-inter)>0 else 0.

    u_vi = np.round(u_p).astype(int); v_vi = np.round(v_p).astype(int)
    in_img2 = valid & (u_vi>=0)&(u_vi<Wm)&(v_vi>=0)&(v_vi<Hm)
    in_msk  = np.zeros(len(verts_r3), dtype=bool)
    in_msk[in_img2] = binary_mask[v_vi[in_img2], u_vi[in_img2]]
    n_in_mask = int(in_msk.sum())
    coverage  = n_in_mask / max(1, int(valid_proj.sum()))

    ctr   = verts_r3.mean(axis=0)
    dims  = verts_r3.max(axis=0) - verts_r3.min(axis=0)
    Z_front = float(verts_r3[:,2].min())
    Z_front_err = Z_front - Z_lidar_front if Z_lidar_front is not None else float('nan')
    dim_err = float(np.abs(dims - CAR_PRIOR_DIMS).mean())
    ctr_ego = R_cam2ego @ ctr + t_cam2ego

    return {'label':label,'iou':iou,'coverage':coverage,'Z_front':Z_front,
            'Z_front_err':Z_front_err,'dims':dims,'ctr':ctr,'ctr_ego':ctr_ego,
            'dim_err':dim_err,'n_in_mask':n_in_mask,
            'bbox_mesh':[bx1,by1,bx2,by2]}


# ── Hybrid anchor helper ─────────────────────────────────────────────────────
def apply_hybrid_anchor(verts_r3, binary_mask, K_proj, u_in, v_in, Z_in, search_radius=8):
    Hm, Wm = binary_mask.shape
    fx_p=float(K_proj[0,0]); fy_p=float(K_proj[1,1])
    cx_p=float(K_proj[0,2]); cy_p=float(K_proj[1,2])

    valid_z = verts_r3[:,2] > 1e-6
    u_v = np.where(valid_z, fx_p*verts_r3[:,0]/np.maximum(verts_r3[:,2],1e-6)+cx_p, -1.)
    v_v = np.where(valid_z, fy_p*verts_r3[:,1]/np.maximum(verts_r3[:,2],1e-6)+cy_p, -1.)
    u_vi2=np.round(u_v).astype(int); v_vi2=np.round(v_v).astype(int)
    in_img_v=valid_z&(u_vi2>=0)&(u_vi2<Wm)&(v_vi2>=0)&(v_vi2<Hm)
    mhit=np.zeros(len(verts_r3),dtype=bool)
    mhit[in_img_v]=binary_mask[v_vi2[in_img_v],u_vi2[in_img_v]]
    visible_idx=np.where(mhit)[0]

    Z_front_anch=float(Z_in.min()); clean=Z_in<=Z_front_anch*1.5
    Z_c=Z_in[clean]; u_c=u_in[clean]; v_c=v_in[clean]

    scale=1.0; n_matched=0
    if len(visible_idx)>=10 and len(Z_c)>=3:
        lpix=np.stack([u_c,v_c],axis=1).astype(np.float32)
        vpix=np.stack([u_v[visible_idx],v_v[visible_idx]],axis=1).astype(np.float32)
        tree=cKDTree(lpix); dists,nn_idx=tree.query(vpix,k=1,workers=-1)
        matched=dists<=search_radius
        if matched.sum()<3: matched=dists<=search_radius*3
        if matched.sum()>=3:
            zm=verts_r3[visible_idx[matched],2]; zl=Z_c[nn_idx[matched]]
            ratios=zl/np.maximum(zm,1e-6); scale=float(np.median(ratios)); n_matched=int(matched.sum())

    vs=verts_r3*scale
    Z_med=float(np.median(vs[:,2])); near=vs[:,2]<=Z_med
    near_ctr=(vs[near].mean(axis=0) if near.sum()>=10 else vs.mean(axis=0))
    Xl=(u_c-cx_p)*Z_c/fx_p; Yl=(v_c-cy_p)*Z_c/fy_p
    lidar_ctr=np.array([Xl.mean(),Yl.mean(),Z_c.mean()],dtype=np.float32)
    t=lidar_ctr-near_ctr
    return (vs+t).astype(np.float32), scale, t, len(visible_idx), n_matched


Z_lidar_front_obj = float(Z_obj.min()) if obj_filter_ok and n_valid_obj > 0 else Z_lidar_anchor

print('Computing metrics...')
verts_g, faces_g = mesh_to_r3(output_global)
verts_l, faces_l = mesh_to_r3(output_local)
m_g = compute_metrics(verts_g, binary_mask, K_np, Z_lidar_front_obj, 'Global raw')
m_l = compute_metrics(verts_l, binary_mask, K_np, Z_lidar_front_obj, 'Local  raw')

for m in [m_g, m_l]:
    print(f'\n[{m["label"]}]')
    print(f'  IoU={m["iou"]:.3f}  coverage={m["coverage"]:.3f}  '
          f'n_in_mask={m["n_in_mask"]}')
    print(f'  Z_front={m["Z_front"]:.3f} m  err={m["Z_front_err"]:+.3f} m')
    print(f'  dims={m["dims"][0]:.2f}×{m["dims"][1]:.2f}×{m["dims"][2]:.2f} m  '
          f'dim_err={m["dim_err"]:.3f}')
    print(f'  ctr(R3): X={m["ctr"][0]:+.2f}  Y={m["ctr"][1]:+.2f}  Z={m["ctr"][2]:.2f} m')

In [154]:
# ── Side-by-side: global raw | local raw ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, (label, verts, m) in zip(axes, [
    ('Global raw', verts_g, m_g),
    ('Local  raw', verts_l, m_l),
]):
    img_m = draw_mesh_on_image(img_rgb, verts, K_np, color=(255,100,0))
    ax.imshow(img_m[yt:yb, xl:xr])

    # Mesh projected bbox
    bb = m['bbox_mesh']
    ax.add_patch(plt.Rectangle((bb[0]-xl,bb[1]-yt),bb[2]-bb[0],bb[3]-bb[1],
                                lw=2,edgecolor='red',facecolor='none',label='mesh bbox'))
    # Mask bbox
    ax.add_patch(plt.Rectangle((xs_m.min()-xl,ys_m.min()-yt),
                                xs_m.max()-xs_m.min(),ys_m.max()-ys_m.min(),
                                lw=2,edgecolor='lime',facecolor='none',label='mask bbox'))
    # Filtered LiDAR
    if n_valid_obj > 0:
        ax.scatter(u_obj-xl, v_obj-yt, c=Z_obj, cmap='cool',
                   s=30, alpha=0.7, zorder=6, label='LiDAR (filtered)')

    ax.set_title(f'{label}\nIoU={m["iou"]:.3f}  cov={m["coverage"]:.3f}  '
                 f'Z_front_err={m["Z_front_err"]:+.3f} m\n'
                 f'dims={m["dims"][0]:.2f}×{m["dims"][1]:.2f}×{m["dims"][2]:.2f} m')
    ax.legend(loc='upper right', fontsize=7)
    ax.axis('off')

plt.suptitle(f'Option C raw meshes — frame {FRAME_IDX} | obj [{OBJ_IDX}]', fontsize=12)
plt.tight_layout(); plt.show()

## §9. Smart post-hoc correction

Apply hybrid anchor (v3 scale + v2 translation) **only if it improves at least one objective metric**
by a meaningful margin. Metrics checked: bbox IoU (+0.02), mask coverage (+0.02),
Z_front error (−0.10 m abs), dim error (−0.05 m).

This avoids over-correcting a mesh that is already well-placed.

In [155]:
def maybe_apply_posthoc(verts_r3, binary_mask, K, u_in, v_in, Z_in,
                          Z_lidar_front, base_metrics, label):
    if len(Z_in) < 3:
        print(f'[{label}] Skipped — insufficient LiDAR ({len(Z_in)} pts)')
        return verts_r3, False, base_metrics

    try:
        vh, scale_h, t_h, n_vis, n_mat = apply_hybrid_anchor(
            verts_r3, binary_mask, K, u_in, v_in, Z_in)
    except Exception as e:
        print(f'[{label}] Hybrid anchor failed: {e}')
        return verts_r3, False, base_metrics

    m_h = compute_metrics(vh, binary_mask, K, Z_lidar_front, label=label+'_ph')

    iou_ok  = m_h['iou']      > base_metrics['iou']      + 0.02
    cov_ok  = m_h['coverage'] > base_metrics['coverage']  + 0.02
    zf_ok   = (abs(m_h['Z_front_err']) < abs(base_metrics['Z_front_err']) - 0.10
               and not np.isnan(base_metrics['Z_front_err']))
    dim_ok  = m_h['dim_err']  < base_metrics['dim_err']   - 0.05
    n_ok    = sum([iou_ok, cov_ok, zf_ok, dim_ok])

    improved = (n_ok >= 1)

    status = 'APPLIED' if improved else 'SKIPPED'
    print(f'[{label}] Post-hoc {status} ({n_ok}/4 metrics improved):')
    print(f'  IoU:       {base_metrics["iou"]:.3f} → {m_h["iou"]:.3f}  {"✓" if iou_ok else "—"}')
    print(f'  Coverage:  {base_metrics["coverage"]:.3f} → {m_h["coverage"]:.3f}  {"✓" if cov_ok else "—"}')
    print(f'  Z_front_err: {base_metrics["Z_front_err"]:+.3f} → {m_h["Z_front_err"]:+.3f} m  {"✓" if zf_ok else "—"}')
    print(f'  dim_err:   {base_metrics["dim_err"]:.3f} → {m_h["dim_err"]:.3f}  {"✓" if dim_ok else "—"}')
    if improved:
        print(f'  scale={scale_h:.4f}  ΔX={t_h[0]:+.3f} ΔY={t_h[1]:+.3f} ΔZ={t_h[2]:+.3f} m')
        return vh.astype(np.float32), True, m_h
    return verts_r3, False, base_metrics


# Use filtered object LiDAR for post-hoc (cleaner than all in-mask)
u_ph = u_obj if obj_filter_ok else u_in_mask_f
v_ph = v_obj if obj_filter_ok else v_in_mask_f
Z_ph = Z_obj if obj_filter_ok else Z_in_mask_arr

print('=== Smart post-hoc correction ===')
verts_g_final, ph_applied_g, m_g_final = maybe_apply_posthoc(
    verts_g, binary_mask, K_np, u_ph, v_ph, Z_ph, Z_lidar_front_obj, m_g, 'Global')
print()
verts_l_final, ph_applied_l, m_l_final = maybe_apply_posthoc(
    verts_l, binary_mask, K_np, u_ph, v_ph, Z_ph, Z_lidar_front_obj, m_l, 'Local')

## §10. Baseline: MoGe-only + hybrid post-hoc

Run SAM3D with the pure MoGe pointmap (no metric calibration) as a baseline, then apply the
hybrid post-hoc anchor. This lets us quantify how much Option C's metric conditioning contributes
beyond what post-hoc correction alone achieves.

In [156]:
print('Running SAM3D [MoGe baseline]...')
output_moge = inference_model(rgb, binary_mask, seed=42, pointmap=moge_ptmap_p3d)
print('Done.')

verts_moge, faces_moge = mesh_to_r3(output_moge)
m_moge_raw = compute_metrics(verts_moge, binary_mask, K_np, Z_lidar_front_obj, 'MoGe raw')

print(f'\n[MoGe raw]:')
print(f'  IoU={m_moge_raw["iou"]:.3f}  coverage={m_moge_raw["coverage"]:.3f}')
print(f'  Z_front={m_moge_raw["Z_front"]:.4f} (MoGe units — non-metric)')

verts_moge_ph, ph_moge, m_moge_ph = maybe_apply_posthoc(
    verts_moge, binary_mask, K_np, u_ph, v_ph, Z_ph, Z_lidar_front_obj,
    m_moge_raw, 'MoGe+posthoc')

## §11. Full comparison — all strategies

Four strategies compared:
1. **Global raw**: Option C with global affine (no post-hoc)
2. **Local raw**: Option C with local affine (no post-hoc)
3. **Global final**: Global + smart post-hoc (applied only if improves)
4. **Local final**: Local + smart post-hoc (applied only if improves)
5. **MoGe baseline**: MoGe-only + forced hybrid post-hoc

In [157]:
# ── 5-panel visualization ──────────────────────────────────────────────────────
all_strategies = [
    ('Global raw',   verts_g,       m_g,       'blue'),
    ('Local  raw',   verts_l,       m_l,       'darkorange'),
    ('Global final', verts_g_final, m_g_final, 'steelblue'),
    ('Local  final', verts_l_final, m_l_final, 'chocolate'),
    ('MoGe+posthoc', verts_moge_ph, m_moge_ph, 'gray'),
]

fig, axes = plt.subplots(1, 5, figsize=(30, 7))
for ax, (label, verts, m, color) in zip(axes, all_strategies):
    img_m = draw_mesh_on_image(img_rgb, verts, K_np, color=(255,100,0))
    ax.imshow(img_m[yt:yb, xl:xr])
    bb = m['bbox_mesh']
    ax.add_patch(plt.Rectangle((bb[0]-xl,bb[1]-yt),bb[2]-bb[0],bb[3]-bb[1],
                                lw=2,edgecolor='red',facecolor='none'))
    ax.add_patch(plt.Rectangle((xs_m.min()-xl,ys_m.min()-yt),
                                xs_m.max()-xs_m.min(),ys_m.max()-ys_m.min(),
                                lw=2,edgecolor='lime',facecolor='none'))
    if n_valid_obj > 0:
        ax.scatter(u_obj-xl, v_obj-yt, c='cyan', s=15, alpha=0.6, zorder=6)
    ax.set_title(f'{label}\nIoU={m["iou"]:.3f}  cov={m["coverage"]:.3f}\n'
                 f'Zf_err={m["Z_front_err"]:+.3f}m  dim_err={m["dim_err"]:.3f}',
                 fontsize=8)
    ax.axis('off')

plt.suptitle(f'All strategies — frame {FRAME_IDX} | obj [{OBJ_IDX}]', fontsize=13)
plt.tight_layout(); plt.show()

## §12. Summary metrics table

In [158]:
print('=' * 80)
print(f'Frame {FRAME_IDX}  |  obj [{OBJ_IDX}]  |  prob={det["prob"]:.3f}')
print(f'LiDAR in mask: total={n_hits}  filtered_obj={n_valid_obj}  rejected={n_rejected}')
print(f'LiDAR Z_front (filtered): {Z_lidar_front_obj:.3f} m')
print(f'Affine global: {a_global:.4f}×Z + {b_global:.4f}  R²={r2_g:.4f}  RMSE={rmse_g:.3f}m')
if local_fit_ok:
    print(f'Affine local:  {a_local:.4f}×Z + {b_local:.4f}  R²={r2_l:.4f}  RMSE={rmse_l:.3f}m')
else:
    print(f'Affine local:  UNAVAILABLE (fallback to global)')
print()

header = f'{"Strategy":<20} {"IoU":>6} {"Cov":>6} {"Zf_err(m)":>10} {"dim_err":>8} {"W×H×D (m)":>20} {"Z_ctr(m)":>9}'
print(header)
print('-' * len(header))

best_iou = -1; best_strategy = None
for label, verts, m in [
    ('Global raw',   verts_g,       m_g),
    ('Local  raw',   verts_l,       m_l),
    ('Global final', verts_g_final, m_g_final),
    ('Local  final', verts_l_final, m_l_final),
    ('MoGe+posthoc', verts_moge_ph, m_moge_ph),
]:
    d = m['dims']
    zf_str = f'{m["Z_front_err"]:+.3f}' if not np.isnan(m['Z_front_err']) else '  n/a'
    print(f'{label:<20} {m["iou"]:>6.3f} {m["coverage"]:>6.3f} {zf_str:>10} '
          f'{m["dim_err"]:>8.3f} {d[0]:>5.2f}×{d[1]:.2f}×{d[2]:.2f}    {m["ctr"][2]:>9.2f}')
    if m['iou'] > best_iou:
        best_iou = m['iou']; best_strategy = label

print()
print(f'Best strategy by IoU: [{best_strategy}]  IoU={best_iou:.3f}')
print(f'Expected car dims    : {CAR_PRIOR_DIMS[0]:.1f}×{CAR_PRIOR_DIMS[1]:.1f}×{CAR_PRIOR_DIMS[2]:.1f} m')
print('=' * 80)

## §13. Oriented Bounding Box from local raw mesh

Fit a **gravity-aligned OBB** to the local raw mesh (`verts_l`):
1. Transform vertices to **ego frame** (Z = world-up)
2. PCA of the XY footprint → yaw angle of the car
3. Build the 8-corner box from the three OBB axes
4. Transform corners to **LiDAR frame** and **camera image** for visualization

Same wireframe style and coordinate frame as `visualize_lidar_bboxes.ipynb`.

In [ ]:
# ── OBB helper functions: RANSAC ground estimation + gravity-aligned OBB ─────

def estimate_local_ground(
        xyz_lid, R_l2e, t_l2e,
        obj_center_ego,
        crop_r=8.0,
        excl_r=None,
        bot_frac=0.35,
        ransac_n=200,
        ransac_thresh=0.07,
        min_inliers=15,
        max_tilt_deg=12.0,
):
    """
    Estimate the local ground plane under an object using RANSAC on nearby LiDAR.

    Algorithm
    ---------
    1. Transform the full LiDAR sweep to ego frame (Z = world-up).
    2. BEV-crop to a circle of radius `crop_r` around the object centre.
    3. Exclude a circle of radius `excl_r` (approximate object footprint).
    4. Keep the bottom `bot_frac` of the remaining points by Z.
    5. RANSAC: fit z = a*x + b*y + c on random 3-point samples.
    6. Horizontality guard: plane normal must be within `max_tilt_deg` of vertical.
    7. Returns ground_z = a*cx + b*cy + c at the object centre.

    Parameters
    ----------
    xyz_lid        : (N,3) full LiDAR sweep in *LiDAR* frame
    R_l2e, t_l2e   : lidar->ego rigid transform  (ego Z = world-up)
    obj_center_ego : (3,) object centre in *ego* frame
    crop_r         : BEV crop radius around object [m]
    excl_r         : BEV exclusion radius (object footprint) [m]; None = no excl.
    bot_frac       : keep bottom this fraction of cropped points by Z
    ransac_n       : number of RANSAC iterations
    ransac_thresh  : per-point inlier Z residual threshold [m]
    min_inliers    : minimum inliers required for a valid plane
    max_tilt_deg   : max allowed angle between plane normal and vertical [deg]

    Returns
    -------
    dict with keys:
        success        : bool
        ground_z       : float | None  -- estimated ground height at obj centre (ego Z)
        plane_coeff    : (3,) | None   -- [a, b, c] for  z = a*x + b*y + c
        n_ground_pts   : int           -- points in local crop after BEV/excl filter
        n_low_pts      : int           -- after bot_frac filter
        n_inliers      : int
        inlier_mask    : bool array    -- over the *low* pts array
        ground_pts_ego : (n_low_pts,3) -- low pts used for RANSAC (ego frame)
        rmse           : float | None
        tilt_deg       : float | None
        msg            : str
    """
    # Transform to ego frame
    xyz_ego = (R_l2e @ xyz_lid.T).T + t_l2e   # (N, 3)

    cx, cy = float(obj_center_ego[0]), float(obj_center_ego[1])
    dx  = xyz_ego[:, 0] - cx
    dy  = xyz_ego[:, 1] - cy
    bev = np.sqrt(dx**2 + dy**2)

    # BEV crop + object exclusion
    mask = bev < crop_r
    if excl_r is not None and excl_r > 0:
        mask &= (bev > excl_r)

    pts = xyz_ego[mask]
    n_ground_pts = len(pts)

    if n_ground_pts < min_inliers:
        return dict(success=False, ground_z=None, plane_coeff=None,
                    n_ground_pts=n_ground_pts, n_low_pts=0, n_inliers=0,
                    inlier_mask=None, ground_pts_ego=pts, rmse=None,
                    tilt_deg=None,
                    msg=f'FAIL: only {n_ground_pts} pts in local crop (need {min_inliers})')

    # Keep bottom fraction by Z
    z_cut = float(np.percentile(pts[:, 2], bot_frac * 100))
    low   = pts[:, 2] <= z_cut
    pts_low = pts[low]
    n_low_pts = len(pts_low)

    if n_low_pts < min_inliers:
        return dict(success=False, ground_z=None, plane_coeff=None,
                    n_ground_pts=n_ground_pts, n_low_pts=n_low_pts, n_inliers=0,
                    inlier_mask=None, ground_pts_ego=pts_low, rmse=None,
                    tilt_deg=None,
                    msg=f'FAIL: only {n_low_pts} low pts after bot_frac={bot_frac:.0%}')

    # ── RANSAC ────────────────────────────────────────────────────────────────
    n = len(pts_low)
    rng = np.random.default_rng(42)
    best_mask  = np.zeros(n, dtype=bool)
    best_count = 0

    for _ in range(ransac_n):
        idx = rng.choice(n, 3, replace=False)
        samp = pts_low[idx]
        A_s  = np.c_[samp[:, 0], samp[:, 1], np.ones(3)]
        try:
            coeff_s, _, rank, _ = np.linalg.lstsq(A_s, samp[:, 2], rcond=None)
            if rank < 3:
                continue
        except Exception:
            continue
        z_pred = pts_low[:, 0]*coeff_s[0] + pts_low[:, 1]*coeff_s[1] + coeff_s[2]
        m = np.abs(pts_low[:, 2] - z_pred) < ransac_thresh
        if m.sum() > best_count:
            best_count = int(m.sum())
            best_mask  = m

    if best_count < min_inliers:
        return dict(success=False, ground_z=None, plane_coeff=None,
                    n_ground_pts=n_ground_pts, n_low_pts=n_low_pts,
                    n_inliers=best_count, inlier_mask=best_mask,
                    ground_pts_ego=pts_low, rmse=None, tilt_deg=None,
                    msg=f'FAIL: RANSAC found {best_count} inliers (need {min_inliers})')

    # Refit on all inliers
    pts_in = pts_low[best_mask]
    A_in   = np.c_[pts_in[:, 0], pts_in[:, 1], np.ones(len(pts_in))]
    coeff, _, _, _ = np.linalg.lstsq(A_in, pts_in[:, 2], rcond=None)

    # Recompute inlier mask with final coefficients
    z_pred_all  = pts_low[:, 0]*coeff[0] + pts_low[:, 1]*coeff[1] + coeff[2]
    inlier_mask = np.abs(pts_low[:, 2] - z_pred_all) < ransac_thresh
    n_inliers   = int(inlier_mask.sum())
    rmse = float(np.sqrt(np.mean(
        (pts_low[inlier_mask, 2] - z_pred_all[inlier_mask])**2)))

    # ── Horizontality guard ───────────────────────────────────────────────────
    a, b, c = coeff
    normal   = np.array([-a, -b, 1.0])
    normal  /= np.linalg.norm(normal)
    tilt_deg = float(np.degrees(np.arccos(np.clip(abs(normal[2]), 0, 1))))

    if tilt_deg > max_tilt_deg:
        return dict(success=False, ground_z=None, plane_coeff=coeff,
                    n_ground_pts=n_ground_pts, n_low_pts=n_low_pts,
                    n_inliers=n_inliers, inlier_mask=inlier_mask,
                    ground_pts_ego=pts_low, rmse=rmse, tilt_deg=tilt_deg,
                    msg=f'FAIL: plane tilt {tilt_deg:.1f} deg > {max_tilt_deg} deg limit')

    ground_z = float(a * cx + b * cy + c)

    return dict(success=True, ground_z=ground_z, plane_coeff=coeff,
                n_ground_pts=n_ground_pts, n_low_pts=n_low_pts,
                n_inliers=n_inliers, inlier_mask=inlier_mask,
                ground_pts_ego=pts_low, rmse=rmse, tilt_deg=tilt_deg,
                msg=(f'OK  n_crop={n_ground_pts}  n_low={n_low_pts}  '
                     f'inliers={n_inliers}  RMSE={rmse:.4f}m  '
                     f'tilt={tilt_deg:.1f}deg  ground_z={ground_z:+.3f}m'))


def compute_obb_gravity_aligned(verts_r3, R_c2e, t_c2e,
                                 ground_z=None,
                                 height_pct=(5, 98),
                                 height_clamp=(1.1, 2.4)):
    """
    Fit a gravity-aligned OBB to mesh vertices in R3 camera space.

    Footprint (yaw / length / width)
        PCA of mesh XY footprint in ego frame — unchanged from original.

    Vertical extent
        If `ground_z` is provided (from RANSAC ground estimation):
          robust_height = percentile(ego_Z, height_pct[1])
                        - percentile(ego_Z, height_pct[0])
          clamped to `height_clamp` [m]
          z_bottom = ground_z
          z_top    = ground_z + robust_height
        Otherwise: raw mesh min/max Z (original behaviour).

    Parameters
    ----------
    ground_z     : ego-frame ground height at OBB centre, or None
    height_pct   : (low_pct, high_pct) percentiles for robust mesh height
    height_clamp : (min_h, max_h) plausible vehicle height range [m]
    """
    verts_ego = (R_c2e @ verts_r3.T).T + t_c2e

    xy   = verts_ego[:, :2]
    xy_c = xy - xy.mean(axis=0)
    _, evecs = np.linalg.eigh(np.cov(xy_c.T))
    evecs = evecs[:, ::-1]

    ax_l = np.array([evecs[0, 0], evecs[1, 0], 0.], dtype=np.float64)
    ax_w = np.array([evecs[0, 1], evecs[1, 1], 0.], dtype=np.float64)
    ax_h = np.array([0., 0., 1.], dtype=np.float64)
    if ax_l[0] * ax_w[1] - ax_l[1] * ax_w[0] < 0:
        ax_w = -ax_w
    axes = np.stack([ax_l, ax_w, ax_h], axis=1)

    proj  = verts_ego @ axes
    min_p = proj.min(axis=0).copy()
    max_p = proj.max(axis=0).copy()

    # ── Vertical extent ───────────────────────────────────────────────────────
    if ground_z is not None:
        vz = verts_ego[:, 2]
        raw_h = float(np.percentile(vz, height_pct[1]) - np.percentile(vz, height_pct[0]))
        robust_height = float(np.clip(raw_h, height_clamp[0], height_clamp[1]))
        min_p[2] = ground_z
        max_p[2] = ground_z + robust_height
    # else: keep raw min/max

    mid_p      = (min_p + max_p) / 2
    center_ego = axes @ mid_p

    off = np.array([
        [min_p[0]-mid_p[0], min_p[1]-mid_p[1], min_p[2]-mid_p[2]],
        [max_p[0]-mid_p[0], min_p[1]-mid_p[1], min_p[2]-mid_p[2]],
        [max_p[0]-mid_p[0], max_p[1]-mid_p[1], min_p[2]-mid_p[2]],
        [min_p[0]-mid_p[0], max_p[1]-mid_p[1], min_p[2]-mid_p[2]],
        [min_p[0]-mid_p[0], min_p[1]-mid_p[1], max_p[2]-mid_p[2]],
        [max_p[0]-mid_p[0], min_p[1]-mid_p[1], max_p[2]-mid_p[2]],
        [max_p[0]-mid_p[0], max_p[1]-mid_p[1], max_p[2]-mid_p[2]],
        [min_p[0]-mid_p[0], max_p[1]-mid_p[1], max_p[2]-mid_p[2]],
    ])
    corners_ego = (axes @ off.T).T + center_ego
    dims    = max_p - min_p
    yaw_rad = float(np.arctan2(evecs[1, 0], evecs[0, 0]))
    return corners_ego, center_ego, dims, yaw_rad, axes


print('estimate_local_ground + compute_obb_gravity_aligned defined.')


In [ ]:
# ── Ground-anchored OBB computation ──────────────────────────────────────────
#
# Strategy:
#   1. Preliminary OBB (raw mesh Z) → centre for local crop
#   2. RANSAC ground estimation from surrounding LiDAR
#   3. Final OBB with ground_z anchor + robust mesh height

# Step 1: preliminary OBB (raw) to locate the object
_vl64 = verts_l.astype(np.float64)
_Rc2e = R_cam2ego.astype(np.float64)
_tc2e = t_cam2ego.astype(np.float64)

_corners_prelim, _center_prelim, _dims_prelim, _, _ = compute_obb_gravity_aligned(
    _vl64, _Rc2e, _tc2e)           # no ground_z → raw mesh Z

# Approximate exclusion radius from preliminary footprint
_excl_r = float(max(_dims_prelim[0], _dims_prelim[1])) * 0.55 + 0.3

# Step 2: RANSAC local ground estimation
gnd = estimate_local_ground(
    xyz_lid, R_l2e, t_l2e,
    obj_center_ego = _center_prelim,
    crop_r         = 10.0,
    excl_r         = _excl_r,
    bot_frac       = 0.35,
    ransac_n       = 250,
    ransac_thresh  = 0.07,
    min_inliers    = 15,
    max_tilt_deg   = 12.0,
)
print(f'Ground estimation: {gnd["msg"]}')

if gnd['success']:
    ground_z = gnd['ground_z']
    _gnd_src = 'RANSAC'
else:
    # Fallback: robust mesh Z percentile as rough floor estimate
    _vz = (R_cam2ego.astype(np.float64) @ _vl64.T).T[:, 2] + _tc2e[2]
    ground_z = float(np.percentile(_vz, 8))
    _gnd_src = f'FALLBACK(mesh p8={ground_z:+.3f}m)'
    print(f'WARNING: RANSAC failed -> fallback ground_z={ground_z:+.3f} m')

# Step 3: final ground-anchored OBB
corners_ego, center_ego, obb_dims, obb_yaw, obb_axes = compute_obb_gravity_aligned(
    _vl64, _Rc2e, _tc2e, ground_z=ground_z)

corners_lidar = (R_l2e.T @ (corners_ego - t_l2e).T).T
corners_r3    = (R_ego2cam @ corners_ego.T + t_ego2cam[:, None]).T

# Update obb_yaw used by VAR (earlier cell used a quick estimate; overwrite)
obb_yaw_src = _gnd_src

print(f'\nOBB local raw ({_gnd_src}):')
print(f'  Dims [LxWxH]  : {obb_dims[0]:.2f} x {obb_dims[1]:.2f} x {obb_dims[2]:.2f} m')
print(f'  Yaw           : {np.degrees(obb_yaw):.1f} deg')
print(f'  ground_z (ego): {ground_z:+.3f} m')
print(f'  Center (LiDAR): '
      f'X={corners_lidar.mean(0)[0]:+.2f}  '
      f'Y={corners_lidar.mean(0)[1]:+.2f}  '
      f'Z={corners_lidar.mean(0)[2]:+.2f} m')


In [ ]:
# ── Ground-anchoring diagnostic visualisation ─────────────────────────────────
#
# Left  (BEV ego XY):  local LiDAR crop, RANSAC inliers/outliers, OBB before/after
# Right (Z profile):   Z histogram of crop + ground line + mesh Z range

fig_gnd, axes_gnd = plt.subplots(1, 2, figsize=(16, 7))

# ── Helpers ───────────────────────────────────────────────────────────────────
def _obb_bev_ego(ax, corners_ego, color, ls, lw, label):
    """Plot OBB footprint in ego XY from 8 ego corners."""
    zs = corners_ego[:, 2]
    face = corners_ego[:4] if zs[:4].mean() <= zs[4:].mean() else corners_ego[4:]
    xs = list(face[:, 0]) + [face[0, 0]]
    ys = list(face[:, 1]) + [face[0, 1]]
    ax.plot(xs, ys, color=color, ls=ls, lw=lw, label=label, zorder=5)

# ── Panel 0: BEV ego XY ───────────────────────────────────────────────────────
ax = axes_gnd[0]

pts_low_ego  = gnd['ground_pts_ego']     # all low pts used by RANSAC
inl_mask     = gnd['inlier_mask']        # may be None if RANSAC failed

if len(pts_low_ego) > 0:
    if inl_mask is not None:
        ax.scatter(pts_low_ego[~inl_mask, 0], pts_low_ego[~inl_mask, 1],
                   c='tomato', s=4, alpha=0.5, label='RANSAC outliers', zorder=2)
        ax.scatter(pts_low_ego[inl_mask,  0], pts_low_ego[inl_mask,  1],
                   c='limegreen', s=6, alpha=0.8, label='RANSAC inliers', zorder=3)
    else:
        ax.scatter(pts_low_ego[:, 0], pts_low_ego[:, 1],
                   c='gray', s=4, alpha=0.5, label='low pts (RANSAC failed)', zorder=2)

# Draw exclusion circle
excl_circle = plt.Circle((_center_prelim[0], _center_prelim[1]), _excl_r,
                          color='gold', fill=False, ls=':', lw=1.5,
                          label=f'excl r={_excl_r:.1f}m', zorder=4)
ax.add_patch(excl_circle)

# OBB before (dashed gray)
_obb_bev_ego(ax, _corners_prelim, 'gray',  '--', 1.5, 'OBB raw (no ground)')
# OBB after  (solid orange)
_obb_bev_ego(ax, corners_ego,     'orangered', '-', 2.5, 'OBB ground-anchored')

# Object centre
ax.scatter([_center_prelim[0]], [_center_prelim[1]], c='yellow', s=60, zorder=6, marker='*')

r_show = max(12.0, _excl_r * 3)
ax.set_xlim(_center_prelim[0] - r_show, _center_prelim[0] + r_show)
ax.set_ylim(_center_prelim[1] - r_show, _center_prelim[1] + r_show)
ax.set_aspect('equal', adjustable='box')
ax.grid(alpha=0.3)
ax.set_xlabel('Ego X (m)'); ax.set_ylabel('Ego Y (m)')
ax.set_title(f'BEV — RANSAC ground estimation\n{gnd["msg"]}', fontsize=9)
ax.legend(loc='upper right', fontsize=8, facecolor='k', labelcolor='w', framealpha=0.7)

# ── Panel 1: Z profile ────────────────────────────────────────────────────────
ax2 = axes_gnd[1]

# Full local crop Z histogram
_xyz_ego_all = (R_l2e @ xyz_lid.T).T + t_l2e
_dx = _xyz_ego_all[:, 0] - _center_prelim[0]
_dy = _xyz_ego_all[:, 1] - _center_prelim[1]
_crop_all = _xyz_ego_all[np.sqrt(_dx**2 + _dy**2) < 10.0]

if len(_crop_all) > 0:
    z_min_hist = float(np.percentile(_crop_all[:, 2], 1)) - 0.3
    z_max_hist = float(np.percentile(_crop_all[:, 2], 99)) + 0.3
    bins = np.linspace(z_min_hist, z_max_hist, 60)
    ax2.hist(_crop_all[:, 2], bins=bins, color='steelblue', alpha=0.4,
             label='local crop (10m)', orientation='horizontal')

if len(pts_low_ego) > 0:
    ax2.hist(pts_low_ego[:, 2], bins=60, color='gray', alpha=0.7,
             label=f'low pts (bot {gnd["n_low_pts"]})', orientation='horizontal')
    if inl_mask is not None and inl_mask.sum() > 0:
        ax2.hist(pts_low_ego[inl_mask, 2], bins=40, color='limegreen', alpha=0.9,
                 label=f'RANSAC inliers ({gnd["n_inliers"]})', orientation='horizontal')

# Ground line
if gnd['ground_z'] is not None:
    ax2.axhline(gnd['ground_z'], color='gold', lw=2.5, ls='--',
                label=f'ground_z = {gnd["ground_z"]:+.3f} m')

# Mesh Z range
_vz_ego = (R_cam2ego.astype(np.float64) @ verts_l.astype(np.float64).T).T[:, 2] + t_cam2ego[2]
ax2.axhline(float(np.percentile(_vz_ego, 5)),  color='orangered', lw=1.5, ls=':',
            label=f'mesh p5={np.percentile(_vz_ego,5):+.3f}m')
ax2.axhline(float(np.percentile(_vz_ego, 98)), color='orangered', lw=1.5, ls='-.',
            label=f'mesh p98={np.percentile(_vz_ego,98):+.3f}m')
ax2.axhline(corners_ego[:, 2].min(), color='orangered', lw=2.5, ls='-',
            label=f'OBB bottom={corners_ego[:,2].min():+.3f}m')
ax2.axhline(corners_ego[:, 2].max(), color='darkorange',  lw=2.5, ls='-',
            label=f'OBB top={corners_ego[:,2].max():+.3f}m')

ax2.set_xlabel('Count')
ax2.set_ylabel('Ego Z (m)')
ax2.set_title(f'Z profile\n'
              f'OBB H={obb_dims[2]:.2f}m  '
              f'(raw mesh p5={np.percentile(_vz_ego,5):+.2f}m  '
              f'p98={np.percentile(_vz_ego,98):+.2f}m)',
              fontsize=9)
ax2.legend(loc='upper right', fontsize=8)
ax2.grid(alpha=0.3, axis='y')

plt.suptitle(f'Ground anchoring diagnostic — Scene {NUSCENES_SCENE_IDX} / Frame {FRAME_IDX}',
             fontsize=11)
plt.tight_layout()
plt.show()


In [160]:
# ── OBB wireframe projected onto the camera image ────────────────────────────
BBOX_EDGES = [
    (0,1),(1,2),(2,3),(3,0),  # bottom face
    (4,5),(5,6),(6,7),(7,4),  # top face
    (0,4),(1,5),(2,6),(3,7),  # vertical pillars
]

def project_corners(corners_r3, K):
    """Project (8,3) R3 corners to (8,2) pixel coords. Returns None for behind-cam."""
    valid = corners_r3[:, 2] > 1e-3
    u = K[0,0]*corners_r3[:,0]/np.maximum(corners_r3[:,2],1e-6) + K[0,2]
    v = K[1,1]*corners_r3[:,1]/np.maximum(corners_r3[:,2],1e-6) + K[1,2]
    uv = np.stack([u, v], axis=1)
    uv[~valid] = np.nan
    return uv

uv_corners = project_corners(corners_r3, K_np)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
pad = 100

for ax, do_zoom in zip(axes, [False, True]):
    ax.imshow(img_rgb)

    # Draw mesh vertices as faint dots
    img_m = draw_mesh_on_image(img_rgb, verts_l, K_np, color=(255, 140, 0))
    ax.imshow(img_m)

    # Draw OBB edges
    for a, b in BBOX_EDGES:
        ua, va = uv_corners[a]
        ub, vb = uv_corners[b]
        if np.isnan(ua) or np.isnan(ub):
            continue
        # Bottom face red, top face blue, pillars white
        edge_idx = BBOX_EDGES.index((a,b))
        color = ('red' if edge_idx < 4 else
                 'deepskyblue' if edge_idx < 8 else 'white')
        lw = 2.5
        ax.plot([ua, ub], [va, vb], color=color, lw=lw, zorder=8)

    # Draw corner dots
    for i, (u_c, v_c) in enumerate(uv_corners):
        if np.isnan(u_c): continue
        ax.scatter([u_c], [v_c], c='yellow', s=40, zorder=10, linewidths=0.5,
                   edgecolors='black')

    # Mask outline
    contours, _ = cv2.findContours(binary_mask.astype(np.uint8),
                                    cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for cnt in contours:
        cp = cnt[:, 0, :]
        ax.plot(np.append(cp[:,0], cp[0,0]), np.append(cp[:,1], cp[0,1]),
                color='lime', lw=1.5, zorder=7)

    # Filtered LiDAR
    if n_valid_obj > 0:
        ax.scatter(u_obj, v_obj, c=Z_obj, cmap='cool', s=25, alpha=0.8,
                   zorder=9, label='LiDAR (filtered)')

    ax.legend(loc='upper right', fontsize=8)
    ax.axis('off')

    if do_zoom:
        ax.set_xlim(xl, xr); ax.set_ylim(yb, yt)
        ax.set_title(f'Zoom — OBB projected  '
                     f'{obb_dims[0]:.2f}×{obb_dims[1]:.2f}×{obb_dims[2]:.2f} m')
    else:
        ax.set_title(f'Full frame — OBB from local raw mesh  '
                     f'(yaw={np.degrees(obb_yaw):.1f}°)')

# Legend patches
from matplotlib.lines import Line2D
axes[1].legend(handles=[
    Line2D([0],[0], color='red',        lw=2, label='OBB bottom face'),
    Line2D([0],[0], color='deepskyblue',lw=2, label='OBB top face'),
    Line2D([0],[0], color='white',      lw=2, label='OBB pillars'),
    Line2D([0],[0], color='lime',       lw=1.5, label='SAM3 mask'),
], loc='upper right', fontsize=8, facecolor='black', labelcolor='white')

plt.suptitle(f'OBB on camera image — frame {FRAME_IDX} | obj [{OBJ_IDX}]', fontsize=12)
plt.tight_layout(); plt.show()


In [161]:
import plotly.graph_objects as go

# ── LiDAR point cloud (full sweep, LiDAR frame) ───────────────────────────────
xyz_lid = pts_lidar.T.astype(np.float64)   # (N, 3) — already in LiDAR frame

# ── Also transform in-mask filtered LiDAR to LiDAR frame ─────────────────────
# u_obj/v_obj are image pixels; back-project via Z_obj and K_np to get 3D in cam
if n_valid_obj > 0:
    u_obj_arr = u_obj.astype(np.float64)
    v_obj_arr = v_obj.astype(np.float64)
    X_obj_r3  = (u_obj_arr - cx) * Z_obj / fx
    Y_obj_r3  = (v_obj_arr - cy) * Z_obj / fy
    pts_obj_r3   = np.stack([X_obj_r3, Y_obj_r3, Z_obj], axis=1)  # (N,3) cam
    pts_obj_ego  = (R_cam2ego @ pts_obj_r3.T).T + t_cam2ego
    pts_obj_lidar = (R_l2e.T @ (pts_obj_ego - t_l2e).T).T         # (N,3) LiDAR

BBOX_EDGES_LOCAL = [
    (0,1),(1,2),(2,3),(3,0),
    (4,5),(5,6),(6,7),(7,4),
    (0,4),(1,5),(2,6),(3,7),
]

traces = []

# ── LiDAR cloud (coloured by height Z) ───────────────────────────────────────
traces.append(go.Scatter3d(
    x=xyz_lid[:,0], y=xyz_lid[:,1], z=xyz_lid[:,2],
    mode='markers',
    marker=dict(size=1, color=xyz_lid[:,2], colorscale='Turbo',
                opacity=0.4, showscale=False),
    name='LiDAR', hoverinfo='skip',
))

# ── Filtered in-mask LiDAR (highlighted) ─────────────────────────────────────
if n_valid_obj > 0:
    traces.append(go.Scatter3d(
        x=pts_obj_lidar[:,0], y=pts_obj_lidar[:,1], z=pts_obj_lidar[:,2],
        mode='markers',
        marker=dict(size=4, color='cyan', opacity=0.9, symbol='circle'),
        name=f'LiDAR in mask (filtered, n={n_valid_obj})',
    ))

# ── OBB wireframe ─────────────────────────────────────────────────────────────
xs, ys, zs = [], [], []
for a, b in BBOX_EDGES_LOCAL:
    xs += [corners_lidar[a,0], corners_lidar[b,0], None]
    ys += [corners_lidar[a,1], corners_lidar[b,1], None]
    zs += [corners_lidar[a,2], corners_lidar[b,2], None]

traces.append(go.Scatter3d(
    x=xs, y=ys, z=zs,
    mode='lines',
    line=dict(color='rgb(255,100,0)', width=5),
    name=f'OBB local raw  {obb_dims[0]:.2f}×{obb_dims[1]:.2f}×{obb_dims[2]:.2f} m',
))

# ── Corner labels ─────────────────────────────────────────────────────────────
traces.append(go.Scatter3d(
    x=corners_lidar[:,0], y=corners_lidar[:,1], z=corners_lidar[:,2],
    mode='text+markers',
    text=[str(i) for i in range(8)],
    marker=dict(size=3, color='yellow'),
    textfont=dict(size=10, color='yellow'),
    name='OBB corners',
    showlegend=False,
))

# ── OBB forward direction arrow (long axis from center) ───────────────────────
ctr_lid = corners_lidar.mean(axis=0)
# Long axis direction in LiDAR frame
ax_l_ego = obb_axes[:, 0]                                   # long axis in ego
ax_l_lid  = R_l2e.T @ ax_l_ego                              # rotate to LiDAR
tip_lid   = ctr_lid + ax_l_lid * (obb_dims[0] / 2 + 0.3)   # tip past box end

traces.append(go.Scatter3d(
    x=[ctr_lid[0], tip_lid[0]], y=[ctr_lid[1], tip_lid[1]], z=[ctr_lid[2], tip_lid[2]],
    mode='lines+markers',
    line=dict(color='yellow', width=6),
    marker=dict(size=[2, 8], color='yellow', symbol=['circle', 'diamond']),
    name='OBB long axis',
    showlegend=True,
))

fig = go.Figure(data=traces)
ctr = ctr_lid
r   = max(obb_dims.max() * 2.5, 8.0)   # zoom radius around OBB
fig.update_layout(
    title=f'Scene {NUSCENES_SCENE_IDX} / Frame {FRAME_IDX} — LiDAR + OBB (local raw mesh)<br>'
          f'L={obb_dims[0]:.2f}m  W={obb_dims[1]:.2f}m  H={obb_dims[2]:.2f}m  '
          f'yaw={np.degrees(obb_yaw):.1f}°',
    scene=dict(
        xaxis=dict(title='X (m)', range=[ctr[0]-r, ctr[0]+r]),
        yaxis=dict(title='Y (m)', range=[ctr[1]-r, ctr[1]+r]),
        zaxis=dict(title='Z (m)',  range=[ctr[2]-r, ctr[2]+r]),
        aspectmode='cube',   # equal visual length for X/Y/Z → OBB stays orthogonal
        camera=dict(eye=dict(x=0, y=-0.5, z=1.8)),
    ),
    margin=dict(l=0, r=0, b=0, t=60),
    height=700,
)
fig.show()


In [162]:
# ── Bird's-eye view in LiDAR frame (matplotlib) ──────────────────────────────
fig, ax = plt.subplots(1, 1, figsize=(12, 10))

# LiDAR points within ±30 m of OBB center, height-colored
dx = np.abs(xyz_lid[:,0] - ctr_lid[0])
dy = np.abs(xyz_lid[:,1] - ctr_lid[1])
nearby = (dx < 30) & (dy < 30)
sc = ax.scatter(xyz_lid[nearby, 0], xyz_lid[nearby, 1],
                c=xyz_lid[nearby, 2], cmap='turbo', s=0.8, alpha=0.5, vmin=-2, vmax=3)
plt.colorbar(sc, ax=ax, label='Z (m)', fraction=0.025)

# In-mask filtered LiDAR
if n_valid_obj > 0:
    ax.scatter(pts_obj_lidar[:,0], pts_obj_lidar[:,1],
               c='cyan', s=30, zorder=6, alpha=0.9, label='LiDAR in mask (filtered)')

# OBB footprint (bottom face = corners 0-3)
foot = [0, 1, 2, 3, 0]
ax.plot(corners_lidar[foot, 0], corners_lidar[foot, 1],
        color='orangered', lw=2.5, zorder=8, label='OBB footprint')

# Mark corners
for i in range(4):
    ax.text(corners_lidar[i,0], corners_lidar[i,1], str(i),
            color='yellow', fontsize=8, ha='center', va='center', zorder=10,
            bbox=dict(boxstyle='circle,pad=0.15', fc='black', alpha=0.6))

# Long-axis arrow
ax.annotate('', xy=(tip_lid[0], tip_lid[1]), xytext=(ctr_lid[0], ctr_lid[1]),
            arrowprops=dict(arrowstyle='->', color='yellow', lw=2.5), zorder=11)

ax.set_xlabel('X_lidar (m)  ← forward'); ax.set_ylabel('Y_lidar (m)  ← left')
ax.set_aspect('equal')
ax.set_xlim(ctr_lid[0] - 15, ctr_lid[0] + 15)
ax.set_ylim(ctr_lid[1] - 15, ctr_lid[1] + 15)
ax.legend(loc='upper right', fontsize=9)
ax.set_title(f"Bird's-eye view (LiDAR frame) — frame {FRAME_IDX} | obj [{OBJ_IDX}]\n"
             f'OBB: {obb_dims[0]:.2f}×{obb_dims[1]:.2f}×{obb_dims[2]:.2f} m  '
             f'yaw={np.degrees(obb_yaw):.1f}°', fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


## §15. Visibility-Aware Refinement (VAR) of local raw mesh

Use the **visible mesh subset** (verts projecting inside a dilated mask) as the
alignment target — not the full mesh bbox.

Corrections applied in order:
1. **Z-front** — shift all verts in camera depth so the nearest visible vert aligns
   with the nearest filtered LiDAR point.
2. **XY ego translation** — slide visible-vert centroid onto LiDAR centroid in the
   horizontal ego plane (clamped <= `max_trans_m`).
3. **Yaw** (optional) — rotate in ego XY via LiDAR-cluster PCA;
   180-degree disambiguation against current OBB yaw; clamped +/- `max_yaw_deg`.
4. **Scale** (optional) — from 90th-pctile LiDAR/vis-mesh radius ratio,
   **hard-clamped [0.95, 1.05]**.

Primary metric: **LiDAR-to-visible-mesh distance** (lower = better surface alignment).

In [163]:
# ── §15 core: z-buffer visibility extraction + VAR refinement ────────────────

def _extract_visible_verts(verts_r3, binary_mask, K,
                            dilation_px=8, zbuf_tol=0.30):
    """
    Extract front-facing mesh vertices via projection + z-buffer.

    Steps
    -----
    1. Project all vertices into image, drop behind-camera / out-of-bounds.
    2. Dilate the binary mask.
    3. Keep only vertices whose pixel falls inside the dilated mask.
    4. Build a per-pixel depth buffer (nearest depth wins).
    5. Keep vertices within zbuf_tol of their pixel's nearest depth
       → removes back-face / occluded vertices.

    Returns
    -------
    vis_mask : (V,) bool  — final visible vertices
    dbg      : dict       — intermediate arrays for debug plots
    """
    H, W = binary_mask.shape
    fx = float(K[0,0]); fy = float(K[1,1])
    cx = float(K[0,2]); cy = float(K[1,2])
    v = verts_r3.astype(np.float64)

    # Project
    vz   = v[:,2] > 1e-6
    u_f  = np.where(vz, fx*v[:,0]/np.maximum(v[:,2],1e-6)+cx, -9999.)
    v_f  = np.where(vz, fy*v[:,1]/np.maximum(v[:,2],1e-6)+cy, -9999.)
    u_vi = np.round(u_f).astype(int)
    v_vi = np.round(v_f).astype(int)
    in_img = vz & (u_f > -9000) & (u_vi >= 0) & (u_vi < W) & (v_vi >= 0) & (v_vi < H)
    idx = np.where(in_img)[0]

    # Dilated mask
    kern  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (dilation_px*2+1, dilation_px*2+1))
    mdil  = cv2.dilate(binary_mask.astype(np.uint8), kern).astype(bool)
    in_dil = np.zeros(len(v), bool)
    if len(idx):
        in_dil[idx] = mdil[v_vi[idx], u_vi[idx]]

    # Per-pixel depth buffer (min depth per pixel across all in-image verts)
    depth_buf = np.full((H, W), np.inf, dtype=np.float64)
    if len(idx):
        np.minimum.at(depth_buf, (v_vi[idx], u_vi[idx]), v[idx, 2])

    # Z-buffer visibility: vert depth ≈ nearest depth at its pixel
    zbuf_vis = np.zeros(len(v), bool)
    if len(idx):
        zbuf_vis[idx] = (
            np.abs(v[idx, 2] - depth_buf[v_vi[idx], u_vi[idx]]) <= zbuf_tol
        )

    vis_mask = in_dil & zbuf_vis
    dbg = dict(u_f=u_f, v_f=v_f, u_vi=u_vi, v_vi=v_vi,
               in_img=in_img, in_dil=in_dil, zbuf_vis=zbuf_vis,
               depth_buf=depth_buf, vis_mask=vis_mask)
    return vis_mask, dbg


# ── Updated distance helper (uses z-buffer + trimmed Chamfer) ─────────────────
def compute_lidar_vis_mesh_dist(verts_r3, binary_mask, K,
                                u_lf, v_lf, Z_lf, R_c2e, t_c2e,
                                dilation_px=8, zbuf_tol=0.30, trim_frac=0.80):
    # Trimmed mean distance: filtered LiDAR → nearest z-buffer-visible mesh vert.
    if len(Z_lf) == 0:
        return float('nan'), 0
    vis_mask, _ = _extract_visible_verts(verts_r3, binary_mask, K, dilation_px, zbuf_tol)
    n_vis = int(vis_mask.sum())
    if n_vis < 3:
        return float('nan'), 0
    R = R_c2e.astype(np.float64); t = t_c2e.astype(np.float64)
    fx_ = float(K[0,0]); fy_ = float(K[1,1])
    cx_ = float(K[0,2]); cy_ = float(K[1,2])
    vis_ego = (R @ verts_r3.astype(np.float64)[vis_mask].T).T + t
    Ulf = u_lf.astype(np.float64); Vlf = v_lf.astype(np.float64); Zlf = Z_lf.astype(np.float64)
    li_ego = (R @ np.stack([(Ulf-cx_)*Zlf/fx_, (Vlf-cy_)*Zlf/fy_, Zlf], 1).T).T + t
    dists, _ = cKDTree(vis_ego).query(li_ego)
    n_keep = max(3, int(len(dists) * trim_frac))
    return float(np.mean(np.sort(dists)[:n_keep])), n_vis


# ── Main VAR function ──────────────────────────────────────────────────────────
def visibility_aware_refinement(
    verts_r3, binary_mask, K,
    u_lf, v_lf, Z_lf,
    R_c2e, t_c2e,
    current_obb_yaw,
    dilation_px=8,
    zbuf_tol=0.30,
    max_trans_xy_m=1.0,
    max_trans_z_m=0.75,
    max_yaw_deg=10.0,
    scale_min=0.95, scale_max=1.05,
    min_vis_verts=15,
    min_lidar_yaw=6,
    yaw_eigenratio_min=2.0,
    trim_frac=0.80,
    do_yaw=True,
    do_scale=False,
    debug=True,
):
    # Visibility-aware post-hoc refinement.  Returns (verts_corrected_f32, info_dict).
    H_, W_ = binary_mask.shape
    fx_ = float(K[0,0]); fy_ = float(K[1,1])
    cx_ = float(K[0,2]); cy_ = float(K[1,2])
    R = R_c2e.astype(np.float64); t = t_c2e.astype(np.float64)

    info = dict(n_vis_verts=0, n_lidar=len(Z_lf),
                delta_cam_z=0., delta_ego_xy=np.zeros(2),
                delta_yaw_deg=0., scale=1.0,
                lidar_mesh_dist=float('nan'),
                lidar_mesh_dist_before=float('nan'),
                status='ok', _dbg=None)

    verts = verts_r3.astype(np.float64)

    # ── 1. Z-buffer visible verts ─────────────────────────────────────────────
    vis_mask, dbg0 = _extract_visible_verts(verts, binary_mask, K, dilation_px, zbuf_tol)
    n_vis = int(vis_mask.sum())
    info['n_vis_verts'] = n_vis
    if n_vis < min_vis_verts:
        info['status'] = f'skip_too_few_vis ({n_vis})'
        return verts_r3.astype(np.float32), info
    if len(Z_lf) < 3:
        info['status'] = f'skip_too_few_lidar ({len(Z_lf)})'
        return verts_r3.astype(np.float32), info

    # ── 2. LiDAR → ego ────────────────────────────────────────────────────────
    Ulf = u_lf.astype(np.float64); Vlf = v_lf.astype(np.float64); Zlf = Z_lf.astype(np.float64)
    li_cam = np.stack([(Ulf-cx_)*Zlf/fx_, (Vlf-cy_)*Zlf/fy_, Zlf], axis=1)
    li_ego = (R @ li_cam.T).T + t                                   # (N, 3)

    # ── 3. Z-front correction in camera depth ─────────────────────────────────
    z_front_li  = float(Zlf.min())
    z_front_vis = float(verts[vis_mask, 2].min())
    dz = float(np.clip(z_front_li - z_front_vis, -max_trans_z_m, max_trans_z_m))
    info['delta_cam_z'] = dz
    verts[:, 2] += dz

    # ── 4. Trimmed XY translation in ego ──────────────────────────────────────
    # Use NN matching (not plain centroid) for robustness
    vis_ego = (R @ verts[vis_mask].T).T + t                          # (n_vis, 3)
    dists_l, nn_idx_l = cKDTree(vis_ego).query(li_ego)
    n_keep = max(3, int(len(dists_l) * trim_frac))
    keep = np.argsort(dists_l)[:n_keep]
    li_trimmed   = li_ego[keep]
    mesh_trimmed = vis_ego[nn_idx_l[keep]]
    dxy = li_trimmed.mean(0)[:2] - mesh_trimmed.mean(0)[:2]
    mag = float(np.linalg.norm(dxy))
    if mag > max_trans_xy_m:
        dxy = dxy * max_trans_xy_m / mag
    info['delta_ego_xy'] = dxy

    # Record "before" dist using trimmed Chamfer on original+Z-shifted vis verts
    dists_before_trim = dists_l  # already computed with Z-shifted verts
    dist_before = float(np.mean(np.sort(dists_before_trim)[:n_keep]))
    info['lidar_mesh_dist_before'] = dist_before

    # Store debug data (before XY correction)
    if debug:
        info['_dbg'] = dict(
            dbg0=dbg0,                          # projection / z-buffer arrays
            li_ego=li_ego,
            vis_ego_before=vis_ego.copy(),
            keep_idx=keep,
            nn_idx_l=nn_idx_l,
            n_keep=n_keep,
        )

    # Apply XY translation
    ve = (R @ verts.T).T + t
    ve[:, :2] += dxy
    verts = (R.T @ (ve - t).T).T

    # ── 5. Yaw correction in ego XY ───────────────────────────────────────────
    dyaw = 0.0
    if do_yaw and len(Zlf) >= min_lidar_yaw:
        li_xy_c = li_ego[:, :2] - li_ego[:, :2].mean(0)
        if li_xy_c.shape[0] >= 2:
            eigvals_li, evecs_li = np.linalg.eigh(np.cov(li_xy_c.T))
            eigenratio = float(eigvals_li[-1] / max(eigvals_li[0], 1e-9))
            if eigenratio >= yaw_eigenratio_min:
                ev = evecs_li[:, -1]
                y1 = np.arctan2(ev[1], ev[0])
                y2 = y1 + np.pi if y1 <= 0 else y1 - np.pi
                diffs = [((c - current_obb_yaw) + np.pi) % (2*np.pi) - np.pi
                         for c in (y1, y2)]
                dyaw = float(min(diffs, key=abs))
                dyaw = float(np.clip(dyaw, -np.radians(max_yaw_deg), np.radians(max_yaw_deg)))
            else:
                info['status'] = info['status'] + '|yaw_skipped_flat_cluster'

    info['delta_yaw_deg'] = float(np.degrees(dyaw))
    if abs(dyaw) > 1e-6:
        ve  = (R @ verts.T).T + t
        cr  = ve.mean(0)
        cd, sd = np.cos(dyaw), np.sin(dyaw)
        Ry  = np.array([[cd, -sd, 0.], [sd, cd, 0.], [0., 0., 1.]])
        ve  = (Ry @ (ve - cr).T).T + cr
        verts = (R.T @ (ve - t).T).T

    # ── 6. Conservative scale ─────────────────────────────────────────────────
    scale = 1.0
    if do_scale and len(Zlf) >= 6:
        vf_ego   = (R @ verts[vis_mask].T).T + t
        ctr_vf   = vf_ego.mean(0)
        ctr_li   = li_ego.mean(0)
        vis_ext  = float(np.percentile(np.linalg.norm(vf_ego[:, :2]-ctr_vf[:2], axis=1), 90))
        li_ext   = float(np.percentile(np.linalg.norm(li_ego[:, :2]-ctr_li[:2], axis=1), 90))
        if vis_ext > 0.1:
            scale = float(np.clip(li_ext / vis_ext, scale_min, scale_max))
    info['scale'] = scale
    if abs(scale - 1.0) > 1e-4:
        ve  = (R @ verts.T).T + t
        cs  = ve.mean(0)
        ve  = (ve - cs) * scale + cs
        verts = (R.T @ (ve - t).T).T

    # ── 7. Final distance + rejection check ───────────────────────────────────
    vis_mask_fin, _ = _extract_visible_verts(verts, binary_mask, K, dilation_px, zbuf_tol)
    n_vis_fin = int(vis_mask_fin.sum())
    if n_vis_fin >= 3:
        vis_ego_fin = (R @ verts[vis_mask_fin].T).T + t
        d_fin, _ = cKDTree(vis_ego_fin).query(li_ego)
        n_k = max(3, int(len(d_fin) * trim_frac))
        dist_after = float(np.mean(np.sort(d_fin)[:n_k]))
    else:
        dist_after = float('inf')
    info['lidar_mesh_dist'] = dist_after

    # Reject if correction made alignment worse (allow 5% slack)
    if np.isfinite(dist_before) and dist_after > dist_before * 1.05:
        info['status'] = f'rejected_worse_dist ({dist_before:.3f} -> {dist_after:.3f})'
        return verts_r3.astype(np.float32), info

    # Reject if any single correction hit its bound exactly (suspicious)
    if (abs(abs(dz)  - max_trans_z_m) < 1e-4 or
        abs(mag - max_trans_xy_m) < 1e-4 or
        abs(abs(dyaw) - np.radians(max_yaw_deg)) < 1e-4):
        info['status'] = info['status'] + '|hit_bound'

    return verts.astype(np.float32), info


print('_extract_visible_verts, compute_lidar_vis_mesh_dist, visibility_aware_refinement defined.')
print('Parameters: max_trans_xy=1.0m  max_trans_z=0.75m  max_yaw=10deg  scale=[0.95,1.05]')
print('z-buffer tol=0.30m  trim_frac=0.80  yaw eigenratio_min=2.0')


In [164]:
# ── Debug visualization for one VAR result ────────────────────────────────────
def visualize_var_debug(verts_before, verts_after, info, binary_mask, img_rgb,
                         K, u_lf, v_lf, Z_lf, R_c2e, t_c2e,
                         xl=None, xr=None, yt=None, yb=None,
                         dilation_px=8, zbuf_tol=0.30, label='VAR'):
    """
    6-panel debug plot for one VAR strategy.

    Panel layout (2 rows x 3 cols):
      [0] Image: in-dil-mask verts (gray), z-buf visible (green), LiDAR (cyan)
      [1] Z-buffer depth map + visible vert overlay
      [2] Ego XY: visible verts + LiDAR + trimmed matched pairs
      [3] Image before: mesh + LiDAR overlay
      [4] Image after:  mesh + LiDAR overlay
      [5] BEV before (orange) / after (green) in ego frame
    """
    H, W = binary_mask.shape
    fx_ = float(K[0,0]); fy_ = float(K[1,1])
    cx_ = float(K[0,2]); cy_ = float(K[1,2])
    R = R_c2e.astype(np.float64); t_ = t_c2e.astype(np.float64)

    # Crop bounds
    if xl is None:
        ys_m_, xs_m_ = np.where(binary_mask)
        pad = 80
        xl  = max(0, int(xs_m_.min()) - pad); xr  = min(W, int(xs_m_.max()) + pad)
        yt  = max(0, int(ys_m_.min()) - pad); yb  = min(H, int(ys_m_.max()) + pad)
    crop = np.s_[yt:yb, xl:xr]

    # Re-extract visibility for before/after
    vis_before, dbg = _extract_visible_verts(verts_before.astype(np.float64), binary_mask,
                                              K, dilation_px, zbuf_tol)
    vis_after,  _   = _extract_visible_verts(verts_after.astype(np.float64),  binary_mask,
                                              K, dilation_px, zbuf_tol)

    # LiDAR in ego
    Ulf = u_lf.astype(np.float64); Vlf = v_lf.astype(np.float64); Zlf = Z_lf.astype(np.float64)
    li_cam = np.stack([(Ulf-cx_)*Zlf/fx_, (Vlf-cy_)*Zlf/fy_, Zlf], axis=1)
    li_ego = (R @ li_cam.T).T + t_

    vb64 = verts_before.astype(np.float64)
    vis_ego_before = (R @ vb64[vis_before].T).T + t_

    # Trimmed NN pairs (from info if available)
    dbg_i = info.get('_dbg', None)

    fig, axes = plt.subplots(2, 3, figsize=(24, 14))

    # ── Panel 0: vertex categories on image ───────────────────────────────────
    ax = axes[0, 0]
    ax.imshow(img_rgb[crop])
    in_img  = dbg['in_img']
    in_dil  = dbg['in_dil']
    zb_vis  = dbg['zbuf_vis']
    uf = dbg['u_f']; vf_ = dbg['v_f']

    # All in-image verts (gray)
    ax.scatter(uf[in_img]-xl, vf_[in_img]-yt,
               c='gray', s=1, alpha=0.3, zorder=2, label=f'in-image ({in_img.sum()})')
    # In-dilated-mask but NOT z-buf vis (blue)
    in_dil_only = in_dil & ~zb_vis
    if in_dil_only.any():
        ax.scatter(uf[in_dil_only]-xl, vf_[in_dil_only]-yt,
                   c='royalblue', s=6, alpha=0.6, zorder=3,
                   label=f'in-mask !z-buf ({in_dil_only.sum()})')
    # Z-buffer visible (green)
    if vis_before.any():
        ax.scatter(uf[vis_before]-xl, vf_[vis_before]-yt,
                   c='lime', s=8, alpha=0.8, zorder=4,
                   label=f'z-buf visible ({vis_before.sum()})')
    # LiDAR
    ax.scatter(Ulf-xl, Vlf-yt, c=Zlf, cmap='cool',
               s=40, alpha=0.9, zorder=5, edgecolors='teal', linewidths=0.4,
               label=f'LiDAR ({len(Zlf)})')
    # Dilated mask contour
    kern = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (dilation_px*2+1,)*2)
    mdil = cv2.dilate(binary_mask.astype(np.uint8), kern)
    cnts, _ = cv2.findContours(mdil, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for cnt in cnts:
        cp = cnt[:,0,:]; ax.plot(cp[:,0]-xl, cp[:,1]-yt, 'yellow', lw=1, zorder=6)
    cnts2, _ = cv2.findContours(binary_mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for cnt in cnts2:
        cp = cnt[:,0,:]; ax.plot(cp[:,0]-xl, cp[:,1]-yt, 'lime', lw=1.5, zorder=7)
    ax.set_title(f'Vertex categories\ngray=in-img  blue=in-mask!zbuf  green=zbuf-vis  cyan=LiDAR')
    ax.legend(loc='upper right', fontsize=6, facecolor='black', labelcolor='white')
    ax.axis('off')

    # ── Panel 1: Z-buffer depth map ───────────────────────────────────────────
    ax = axes[0, 1]
    dbuf = dbg['depth_buf']
    dbuf_crop = dbuf[yt:yb, xl:xr].copy()
    finite_mask = np.isfinite(dbuf_crop)
    dbuf_vis = np.zeros_like(dbuf_crop)
    if finite_mask.any():
        dbuf_vis[finite_mask] = dbuf_crop[finite_mask]
    im = ax.imshow(dbuf_vis, cmap='plasma_r', interpolation='nearest')
    plt.colorbar(im, ax=ax, fraction=0.03, label='depth (m)')
    if vis_before.any():
        ax.scatter(dbg['u_f'][vis_before]-xl, dbg['v_f'][vis_before]-yt,
                   c='lime', s=6, alpha=0.7, zorder=4, label='z-buf visible')
    ax.scatter(Ulf-xl, Vlf-yt, c='cyan', s=30, alpha=0.9, zorder=5, label='LiDAR')
    ax.set_title(f'Z-buffer depth map\n(brighter = closer to camera)\nzbuf_tol={zbuf_tol}m')
    ax.legend(fontsize=7); ax.axis('off')

    # ── Panel 2: ego XY — visible mesh + LiDAR + matched pairs ───────────────
    ax = axes[0, 2]
    if vis_ego_before.shape[0] > 0:
        ax.scatter(vis_ego_before[:,0], vis_ego_before[:,1],
                   c='lime', s=8, alpha=0.7, zorder=3, label=f'vis mesh ({len(vis_ego_before)})')
    ax.scatter(li_ego[:,0], li_ego[:,1],
               c='cyan', s=30, alpha=0.9, zorder=4,
               edgecolors='teal', linewidths=0.4, label=f'LiDAR ({len(li_ego)})')

    # Draw trimmed matched pairs as lines
    if dbg_i is not None and vis_ego_before.shape[0] > 0:
        keep = dbg_i['keep_idx']; nn = dbg_i['nn_idx_l']
        vis_e = dbg_i['vis_ego_before']
        n_lines = min(len(keep), 60)
        for ki in keep[:n_lines]:
            lp = li_ego[ki]; mp = vis_e[nn[ki]]
            ax.plot([lp[0], mp[0]], [lp[1], mp[1]],
                    color='yellow', alpha=0.5, lw=0.8, zorder=2)
        ax.scatter(li_ego[keep[:n_lines], 0], li_ego[keep[:n_lines], 1],
                   c='yellow', s=20, zorder=5, label=f'trimmed pairs ({len(keep)})')

    ax.set_aspect('equal'); ax.grid(alpha=0.3)
    ax.set_xlabel('ego X (m)'); ax.set_ylabel('ego Y (m)')
    ax.set_title('Ego XY: vis-mesh + LiDAR + trimmed pairs\n(yellow lines = matched)')
    ax.legend(fontsize=7)

    # ── Panel 3: before image overlay ─────────────────────────────────────────
    ax = axes[1, 0]
    img_b = draw_mesh_on_image(img_rgb, verts_before, K, color=(255, 140, 0))
    ax.imshow(img_b[crop])
    ax.scatter(Ulf-xl, Vlf-yt, c=Zlf, cmap='cool', s=25, alpha=0.8, zorder=5)
    if vis_before.any():
        ax.scatter(dbg['u_f'][vis_before]-xl, dbg['v_f'][vis_before]-yt,
                   c='lime', s=4, alpha=0.6, zorder=6)
    ax.set_title(f'Before ({label})\nli_dist={info["lidar_mesh_dist_before"]:.3f}m')
    ax.axis('off')

    # ── Panel 4: after image overlay ──────────────────────────────────────────
    ax = axes[1, 1]
    img_a = draw_mesh_on_image(img_rgb, verts_after, K, color=(0, 200, 80))
    ax.imshow(img_a[crop])
    ax.scatter(Ulf-xl, Vlf-yt, c=Zlf, cmap='cool', s=25, alpha=0.8, zorder=5)
    if vis_after.any():
        uf_a = np.where(verts_after[:,2]>1e-6,
                        fx_*verts_after[:,0]/np.maximum(verts_after[:,2],1e-6)+cx_, -9999.)
        vf_a = np.where(verts_after[:,2]>1e-6,
                        fy_*verts_after[:,1]/np.maximum(verts_after[:,2],1e-6)+cy_, -9999.)
        ax.scatter(uf_a[vis_after]-xl, vf_a[vis_after]-yt,
                   c='yellow', s=4, alpha=0.6, zorder=6)
    li_s = info['lidar_mesh_dist']
    li_str = f'{li_s:.3f}m' if np.isfinite(li_s) else '-'
    ax.set_title(f'After ({label})\nli_dist={li_str}  '
                 f'dZ={info["delta_cam_z"]:+.2f}m  '
                 f'dXY={np.linalg.norm(info["delta_ego_xy"]):.2f}m\n'
                 f'dyaw={info["delta_yaw_deg"]:+.1f}deg  s={info["scale"]:.3f}')
    ax.axis('off')

    # ── Panel 5: ego XY BEV before / after ────────────────────────────────────
    ax = axes[1, 2]
    va64 = verts_after.astype(np.float64)
    vb_ego = (R @ vb64.T).T + t_
    va_ego = (R @ va64.T).T + t_
    ctr = vb_ego.mean(0)
    r5  = 6.0
    # All mesh verts (before: orange, after: green)
    ax.scatter(vb_ego[:,0], vb_ego[:,1], c='orange', s=1, alpha=0.2, zorder=2, label='mesh before')
    ax.scatter(va_ego[:,0], va_ego[:,1], c='limegreen', s=1, alpha=0.2, zorder=2, label='mesh after')
    # Z-buf vis verts
    if vis_before.any():
        ax.scatter(vis_ego_before[:,0], vis_ego_before[:,1],
                   c='orange', s=12, alpha=0.8, zorder=4, edgecolors='darkorange', lw=0.3)
    if vis_after.any():
        va_vis_ego = (R @ va64[vis_after].T).T + t_
        ax.scatter(va_vis_ego[:,0], va_vis_ego[:,1],
                   c='lime', s=12, alpha=0.8, zorder=4, edgecolors='darkgreen', lw=0.3)
    # LiDAR
    ax.scatter(li_ego[:,0], li_ego[:,1], c='cyan', s=40, alpha=0.9, zorder=5,
               edgecolors='teal', lw=0.4, label='LiDAR')
    ax.set_xlim(ctr[0]-r5, ctr[0]+r5); ax.set_ylim(ctr[1]-r5, ctr[1]+r5)
    ax.set_aspect('equal'); ax.grid(alpha=0.3)
    ax.set_xlabel('ego X (m)'); ax.set_ylabel('ego Y (m)')
    ax.set_title('Ego XY BEV: orange=before  green=after  cyan=LiDAR\n'
                 '(filled=z-buf visible, faint=all mesh)')
    ax.legend(fontsize=7)

    plt.suptitle(f'VAR debug — {label} — frame {FRAME_IDX} | obj [{OBJ_IDX}]\n'
                 f'n_vis={info["n_vis_verts"]}  n_lidar={info["n_lidar"]}  '
                 f'status={info["status"]}',
                 fontsize=11)
    plt.tight_layout()
    plt.show()


print('visualize_var_debug defined.')


In [165]:
# ── Prepare LiDAR input ──────────────────────────────────────────────────────
if obj_filter_ok and n_valid_obj >= 3:
    u_var = u_obj.astype(np.float64)
    v_var = v_obj.astype(np.float64)
    Z_var = Z_obj.astype(np.float64)
    print(f'VAR: depth-window+MAD filtered LiDAR  n={n_valid_obj}  '
          f'Z_front={Z_var.min():.3f} m')
else:
    u_var = v_var = Z_var = np.array([], dtype=np.float64)
    print('No usable filtered LiDAR — VAR will return raw meshes unchanged.')

_Rc2e = R_cam2ego.astype(np.float64)
_tc2e = t_cam2ego.astype(np.float64)
_kw = dict(binary_mask=binary_mask, K=K_np,
           u_lf=u_var, v_lf=v_var, Z_lf=Z_var,
           R_c2e=_Rc2e, t_c2e=_tc2e,
           current_obb_yaw=float(obb_yaw))

vl64 = verts_l.astype(np.float64)

# VAR1: Z-front + XY translation only
verts_var1, info_var1 = visibility_aware_refinement(
    vl64, do_yaw=False, do_scale=False, **_kw)
m_var1 = compute_metrics(verts_var1, binary_mask, K_np, Z_lidar_front_obj, 'VAR1')
dist_var1, _ = compute_lidar_vis_mesh_dist(
    verts_var1, binary_mask, K_np, u_var, v_var, Z_var, _Rc2e, _tc2e)

# VAR2: yaw + Z-front + XY translation
verts_var2, info_var2 = visibility_aware_refinement(
    vl64, do_yaw=True, do_scale=False, **_kw)
m_var2 = compute_metrics(verts_var2, binary_mask, K_np, Z_lidar_front_obj, 'VAR2')
dist_var2, _ = compute_lidar_vis_mesh_dist(
    verts_var2, binary_mask, K_np, u_var, v_var, Z_var, _Rc2e, _tc2e)

# VAR3: yaw + Z-front + XY translation + clamped scale
verts_var3, info_var3 = visibility_aware_refinement(
    vl64, do_yaw=True, do_scale=True, **_kw)
m_var3 = compute_metrics(verts_var3, binary_mask, K_np, Z_lidar_front_obj, 'VAR3')
dist_var3, _ = compute_lidar_vis_mesh_dist(
    verts_var3, binary_mask, K_np, u_var, v_var, Z_var, _Rc2e, _tc2e)

# Baseline distances
dist_raw, _     = compute_lidar_vis_mesh_dist(
    verts_l,       binary_mask, K_np, u_var, v_var, Z_var, _Rc2e, _tc2e)
dist_posthoc, _ = compute_lidar_vis_mesh_dist(
    verts_l_final, binary_mask, K_np, u_var, v_var, Z_var, _Rc2e, _tc2e)

print()
for lbl, iv in [('VAR1', info_var1), ('VAR2', info_var2), ('VAR3', info_var3)]:
    dxy = float(np.linalg.norm(iv['delta_ego_xy']))
    print(f'[{lbl}] status={iv["status"]}  n_vis={iv["n_vis_verts"]}  '
          f'dZ={iv["delta_cam_z"]:+.3f}m  dXY={dxy:.3f}m  '
          f'dyaw={iv["delta_yaw_deg"]:+.1f}deg  scale={iv["scale"]:.3f}  '
          f'dist: {iv["lidar_mesh_dist_before"]:.3f} -> {iv["lidar_mesh_dist"]:.3f}m')

# ── Debug visualization for the best expected strategy (VAR2) ─────────────────
print('\nGenerating debug plot for VAR2 (yaw+Z+trans)...')
visualize_var_debug(
    verts_before=verts_l,
    verts_after=verts_var2,
    info=info_var2,
    binary_mask=binary_mask,
    img_rgb=img_rgb,
    K=K_np,
    u_lf=u_var, v_lf=v_var, Z_lf=Z_var,
    R_c2e=_Rc2e, t_c2e=_tc2e,
    xl=xl, xr=xr, yt=yt, yb=yb,
    label='VAR2: yaw+Z+trans',
)


In [166]:
# ── 5-panel comparison visualization ─────────────────────────────────────────
var_panels = [
    ('Local raw',             verts_l,       m_l,       None,       dist_raw),
    ('VAR1: Z+trans',         verts_var1,    m_var1,    info_var1,  dist_var1),
    ('VAR2: yaw+Z+trans',     verts_var2,    m_var2,    info_var2,  dist_var2),
    ('VAR3: +clamped scale',  verts_var3,    m_var3,    info_var3,  dist_var3),
    ('Hybrid post-hoc (§9)',  verts_l_final, m_l_final, None,       dist_posthoc),
]
fig, axes = plt.subplots(1, 5, figsize=(30, 8))

for ax, (label, verts, m, iv, li_d) in zip(axes, var_panels):
    img_m = draw_mesh_on_image(img_rgb, verts, K_np, color=(255, 100, 0))
    ax.imshow(img_m[yt:yb, xl:xr])
    bb = m['bbox_mesh']
    ax.add_patch(plt.Rectangle((bb[0]-xl, bb[1]-yt), bb[2]-bb[0], bb[3]-bb[1],
                                lw=2, edgecolor='red', facecolor='none'))
    ax.add_patch(plt.Rectangle((xs_m.min()-xl, ys_m.min()-yt),
                                xs_m.max()-xs_m.min(), ys_m.max()-ys_m.min(),
                                lw=2, edgecolor='lime', facecolor='none'))
    if n_valid_obj > 0:
        ax.scatter(u_obj-xl, v_obj-yt, c=Z_obj, cmap='cool', s=25, alpha=0.8, zorder=6)

    li_str = f'{li_d:.3f}m' if li_d is not None and np.isfinite(li_d) else '-'
    tl = [label,
          f'IoU={m["iou"]:.3f}  cov={m["coverage"]:.3f}',
          f'Zf_err={m["Z_front_err"]:+.3f}m  li_dist={li_str}']
    if iv is not None:
        dxy = float(np.linalg.norm(iv['delta_ego_xy']))
        tl += [f'dZ={iv["delta_cam_z"]:+.2f}m  dXY={dxy:.2f}m',
               f'dyaw={iv["delta_yaw_deg"]:+.1f}deg  s={iv["scale"]:.3f}']
    ax.set_title('\n'.join(tl), fontsize=7)
    ax.axis('off')

from matplotlib.patches import Patch
from matplotlib.lines import Line2D
axes[-1].legend(handles=[
    Patch(edgecolor='red',  facecolor='none', lw=2, label='mesh proj bbox'),
    Patch(edgecolor='lime', facecolor='none', lw=2, label='mask bbox'),
    Line2D([0],[0], color='cyan', marker='o', ms=4, ls='none', label='filtered LiDAR'),
], loc='lower right', fontsize=7, facecolor='black', labelcolor='white')

plt.suptitle(f'Visibility-Aware Refinement (VAR) — frame {FRAME_IDX} | obj [{OBJ_IDX}]',
             fontsize=13)
plt.tight_layout()
plt.show()


In [167]:
# ── Metrics comparison table ──────────────────────────────────────────────────
prior_lwh = np.array([CAR_PRIOR_DIMS[2], CAR_PRIOR_DIMS[0], CAR_PRIOR_DIMS[1]])  # [L,W,H]

all_var_rows = [
    ('Local raw',             m_l,       None,       dist_raw),
    ('VAR1: Z+trans',         m_var1,    info_var1,  dist_var1),
    ('VAR2: yaw+Z+trans',     m_var2,    info_var2,  dist_var2),
    ('VAR3: +clamped scale',  m_var3,    info_var3,  dist_var3),
    ('Hybrid post-hoc (§9)',  m_l_final, None,       dist_posthoc),
]

w = 115
print('=' * w)
print(f'Frame {FRAME_IDX}  |  obj [{OBJ_IDX}]  |  Visibility-Aware Refinement')
print(f'Filtered LiDAR: n={n_valid_obj}  Z_front={Z_lidar_front_obj:.3f} m  '
      f'Car prior [L x W x H]: {prior_lwh[0]:.1f} x {prior_lwh[1]:.1f} x {prior_lwh[2]:.1f} m')
print('=' * w)
hdr = (f'{"Strategy":<26} {"IoU":>5} {"Cov":>5} {"Zf_err(m)":>9} '
       f'{"li_dist(m)":>10} {"scale":>6} {"dyaw(deg)":>9} '
       f'{"dZ(m)":>7} {"dXY(m)":>7} {"dim_err":>7}  L x W x H (m)')
print(hdr)
print('-' * w)

for label, m, iv, li_d in all_var_rows:
    d   = m['dims']
    zfe = f'{m["Z_front_err"]:+.3f}' if not np.isnan(m['Z_front_err']) else '    n/a'
    lis = f'{li_d:.3f}' if li_d is not None and np.isfinite(li_d) else '       -'
    if iv is not None:
        scs  = f'{iv["scale"]:.3f}'
        dys  = f'{iv["delta_yaw_deg"]:+.1f}'
        dzs  = f'{iv["delta_cam_z"]:+.3f}'
        dxys = f'{np.linalg.norm(iv["delta_ego_xy"]):.3f}'
    else:
        scs = dys = dzs = dxys = '      -'
    dim_err = float(np.abs(d[[2,0,1]] - prior_lwh).mean()) if len(d) == 3 else float('nan')
    print(f'{label:<26} {m["iou"]:>5.3f} {m["coverage"]:>5.3f} {zfe:>9} '
          f'{lis:>10} {scs:>6} {dys:>9} '
          f'{dzs:>7} {dxys:>7} {dim_err:>7.3f}  {d[0]:.2f} x {d[1]:.2f} x {d[2]:.2f}')

print()
print('li_dist = mean(dist LiDAR pt -> nearest visible vert) in ego frame  [lower = better]')
print('Zf_err  = Z_front_mesh - Z_front_LiDAR  [|lower| = better]')
print('dim_err = mean |predicted - prior [L,W,H]|')


## §14. OBB comparison: Global raw / Local raw / VAR2

Compares three Oriented Bounding Boxes in the same LiDAR 3D scene and Bird's-Eye View:
- **Blue (solid)** — Global affine calibration mesh OBB
- **Orange-red (dashed)** — Local affine calibration mesh OBB (raw)
- **Lime green (dash-dot)** — VAR2 post-hoc refined mesh OBB

Run §15 first so `verts_var2` is available; otherwise only the first two OBBs are shown.


In [ ]:
# ── OBBs from global / VAR2 / VAR3 meshes ────────────────────────────────────
corners_ego_g, center_ego_g, obb_dims_g, obb_yaw_g, obb_axes_g = \
    compute_obb_gravity_aligned(
        verts_g.astype(np.float64), R_cam2ego.astype(np.float64), t_cam2ego.astype(np.float64),
        ground_z=ground_z)
corners_lidar_g = (R_l2e.T @ (corners_ego_g - t_l2e).T).T

# VAR2 OBB
try:
    corners_ego_v2, _, obb_dims_v2, obb_yaw_v2, obb_axes_v2 = \
        compute_obb_gravity_aligned(
            verts_var2.astype(np.float64), R_cam2ego.astype(np.float64), t_cam2ego.astype(np.float64),
            ground_z=ground_z)
    corners_lidar_v2 = (R_l2e.T @ (corners_ego_v2 - t_l2e).T).T
    var2_obb_ok = True
except NameError:
    var2_obb_ok = False
    print('WARNING: verts_var2 not defined — run §10 (VAR) first.')

# VAR3 OBB
try:
    corners_ego_v3, _, obb_dims_v3, obb_yaw_v3, obb_axes_v3 = \
        compute_obb_gravity_aligned(
            verts_var3.astype(np.float64), R_cam2ego.astype(np.float64), t_cam2ego.astype(np.float64),
            ground_z=ground_z)
    corners_lidar_v3 = (R_l2e.T @ (corners_ego_v3 - t_l2e).T).T
    var3_obb_ok = True
except NameError:
    var3_obb_ok = False

# Bounding sphere for zoom centering
_all_c = np.vstack([corners_lidar, corners_lidar_g]
                   + ([corners_lidar_v2] if var2_obb_ok else [])
                   + ([corners_lidar_v3] if var3_obb_ok else []))
ctr_cmp = _all_c.mean(axis=0)
r_cmp   = max(max(obb_dims.max(), obb_dims_g.max(),
                   *(obb_dims_v2.max(),) if var2_obb_ok else (0.,),
                   *(obb_dims_v3.max(),) if var3_obb_ok else (0.,)) * 2.5, 8.0)

# ── Comparison table ──────────────────────────────────────────────────────────
prior_lwh = np.array([CAR_PRIOR_DIMS[2], CAR_PRIOR_DIMS[0], CAR_PRIOR_DIMS[1]])
rows = [
    ('Global raw',  obb_dims_g,  obb_yaw_g,  corners_lidar_g),
    ('Local raw',   obb_dims,    obb_yaw,    corners_lidar),
]
if var2_obb_ok:
    rows.append(('VAR2', obb_dims_v2, obb_yaw_v2, corners_lidar_v2))
if var3_obb_ok:
    rows.append(('VAR3 (+scale)', obb_dims_v3, obb_yaw_v3, corners_lidar_v3))
rows.append(('Car prior', prior_lwh, float('nan'), None))

print(f'{"":20s}  {"L(m)":>6}  {"W(m)":>6}  {"H(m)":>6}  '
      f'{"Yaw":>7}  {"dim_err":>8}  {"ctrZ":>7}')
print('-' * 80)
for lbl, dims, yaw, cl in rows:
    de = float(np.abs(dims - prior_lwh).mean())
    cz = f'{cl.mean(0)[2]:+7.2f}' if cl is not None else '   -- '
    ys = f'{np.degrees(yaw):7.1f}' if np.isfinite(yaw) else '   -- '
    print(f'{lbl:20s}  {dims[0]:6.2f}  {dims[1]:6.2f}  {dims[2]:6.2f}  {ys}  {de:8.3f}  {cz}')

print(f'\nground_z = {ground_z:+.3f} m ({_gnd_src})')


In [ ]:
# ── Plotly 3D: Global / Local / VAR2 / VAR3 OBBs on LiDAR cloud ─────────────
import plotly.graph_objects as go

_EDGES = [(0,1),(1,2),(2,3),(3,0),(4,5),(5,6),(6,7),(7,4),
          (0,4),(1,5),(2,6),(3,7)]

def _obb_wire(corners, dims, label, color):
    xs, ys, zs = [], [], []
    for a, b in _EDGES:
        xs += [corners[a,0], corners[b,0], None]
        ys += [corners[a,1], corners[b,1], None]
        zs += [corners[a,2], corners[b,2], None]
    return go.Scatter3d(x=xs, y=ys, z=zs, mode='lines',
                        line=dict(color=color, width=5),
                        name=f'{label}  {dims[0]:.2f}x{dims[1]:.2f}x{dims[2]:.2f} m')

def _obb_arrow(corners, axes_mat, dims, color, label):
    ctr    = corners.mean(axis=0)
    fwd    = R_l2e.T @ axes_mat[:, 0]
    tip    = ctr + fwd * (dims[0] / 2 + 0.4)
    return go.Scatter3d(
        x=[ctr[0], tip[0]], y=[ctr[1], tip[1]], z=[ctr[2], tip[2]],
        mode='lines+markers',
        line=dict(color=color, width=6),
        marker=dict(size=[2, 8], color=color, symbol=['circle', 'diamond']),
        name=f'{label} fwd')

tr = []
# LiDAR cloud
tr.append(go.Scatter3d(
    x=xyz_lid[:,0], y=xyz_lid[:,1], z=xyz_lid[:,2],
    mode='markers',
    marker=dict(size=1, color=xyz_lid[:,2], colorscale='Turbo', opacity=0.35, showscale=False),
    name='LiDAR', hoverinfo='skip'))

# In-mask LiDAR
if n_valid_obj > 0:
    tr.append(go.Scatter3d(
        x=pts_obj_lidar[:,0], y=pts_obj_lidar[:,1], z=pts_obj_lidar[:,2],
        mode='markers',
        marker=dict(size=4, color='cyan', opacity=0.9),
        name=f'LiDAR in mask ({n_valid_obj})'))

# RANSAC ground inliers (in LiDAR frame)
if gnd['success'] and gnd['inlier_mask'] is not None:
    _gi = gnd['ground_pts_ego'][gnd['inlier_mask']]
    _gi_lid = (R_l2e.T @ (_gi - t_l2e).T).T
    tr.append(go.Scatter3d(
        x=_gi_lid[:,0], y=_gi_lid[:,1], z=_gi_lid[:,2],
        mode='markers',
        marker=dict(size=2, color='limegreen', opacity=0.6),
        name=f'Ground inliers ({gnd["n_inliers"]})'))

# OBBs
tr.append(_obb_wire(corners_lidar_g, obb_dims_g, 'Global raw', 'rgb(30,144,255)'))
tr.append(_obb_arrow(corners_lidar_g, obb_axes_g, obb_dims_g, 'rgb(30,144,255)', 'Global raw'))

tr.append(_obb_wire(corners_lidar,   obb_dims,   'Local raw', 'rgb(255,100,0)'))
tr.append(_obb_arrow(corners_lidar,  obb_axes,   obb_dims,   'rgb(255,100,0)', 'Local raw'))

if var2_obb_ok:
    tr.append(_obb_wire(corners_lidar_v2, obb_dims_v2, 'VAR2', 'rgb(50,220,80)'))
    tr.append(_obb_arrow(corners_lidar_v2, obb_axes_v2, obb_dims_v2, 'rgb(50,220,80)', 'VAR2'))

if var3_obb_ok:
    tr.append(_obb_wire(corners_lidar_v3, obb_dims_v3, 'VAR3 +scale', 'rgb(200,80,220)'))
    tr.append(_obb_arrow(corners_lidar_v3, obb_axes_v3, obb_dims_v3, 'rgb(200,80,220)', 'VAR3 +scale'))

ctr = ctr_cmp; r = r_cmp
fig14 = go.Figure(data=tr)
fig14.update_layout(
    title=(f'Scene {NUSCENES_SCENE_IDX} / Frame {FRAME_IDX} — LiDAR + OBBs<br>'
           f'ground_z={ground_z:+.3f}m ({_gnd_src})'),
    scene=dict(
        xaxis=dict(title='X (m)', range=[ctr[0]-r, ctr[0]+r]),
        yaxis=dict(title='Y (m)', range=[ctr[1]-r, ctr[1]+r]),
        zaxis=dict(title='Z (m)', range=[ctr[2]-r, ctr[2]+r]),
        aspectmode='cube',
        camera=dict(eye=dict(x=0, y=-0.5, z=1.8)),
    ),
    margin=dict(l=0,r=0,b=0,t=60), height=700,
)
fig14.show()


In [ ]:
# ── BEV: Global / Local / VAR2 / VAR3 OBB footprints ────────────────────────
fig_bev, ax_bev = plt.subplots(1, 1, figsize=(9, 9))

def _bev_foot(ax, corners_lidar, color, ls, lw, label):
    zs = corners_lidar[:, 2]
    face = corners_lidar[:4] if zs[:4].mean() <= zs[4:].mean() else corners_lidar[4:]
    xs = list(face[:, 0]) + [face[0, 0]]
    ys = list(face[:, 1]) + [face[0, 1]]
    ax.plot(xs, ys, color=color, ls=ls, lw=lw, label=label, zorder=3)
    cx, cy = corners_lidar[:, 0].mean(), corners_lidar[:, 1].mean()
    ax.scatter([cx], [cy], color=color, s=40, zorder=5)

def _bev_fwd(ax, corners_lidar, axes_mat, dims, color):
    ctr    = corners_lidar.mean(0)[:2]
    fwd    = (R_l2e.T @ axes_mat[:, 0])[:2]
    tip    = ctr + fwd * (dims[0] / 2 + 0.4)
    ax.annotate('', xy=tip, xytext=ctr,
                arrowprops=dict(arrowstyle='->', color=color, lw=2))

# Background LiDAR cloud
ax_bev.scatter(xyz_lid[:, 0], xyz_lid[:, 1],
               c=xyz_lid[:, 2], cmap='Greys', s=1, alpha=0.2, zorder=1)
if n_valid_obj > 0:
    ax_bev.scatter(pts_obj_lidar[:, 0], pts_obj_lidar[:, 1],
                   c='cyan', s=18, alpha=0.85, zorder=4,
                   label=f'LiDAR in mask ({n_valid_obj})')

# RANSAC ground inliers in BEV
if gnd['success'] and gnd['inlier_mask'] is not None:
    _gi = gnd['ground_pts_ego'][gnd['inlier_mask']]
    _gi_lid = (R_l2e.T @ (_gi - t_l2e).T).T
    ax_bev.scatter(_gi_lid[:, 0], _gi_lid[:, 1],
                   c='limegreen', s=3, alpha=0.5, zorder=2,
                   label=f'Ground inliers ({gnd["n_inliers"]})')

# OBB footprints
_bev_foot(ax_bev, corners_lidar_g,
          'dodgerblue', '-',   2.5,
          f'Global  {obb_dims_g[0]:.2f}x{obb_dims_g[1]:.2f}x{obb_dims_g[2]:.2f} m')
_bev_fwd(ax_bev, corners_lidar_g, obb_axes_g, obb_dims_g, 'dodgerblue')

_bev_foot(ax_bev, corners_lidar,
          'orangered', '--',  2.5,
          f'Local  {obb_dims[0]:.2f}x{obb_dims[1]:.2f}x{obb_dims[2]:.2f} m')
_bev_fwd(ax_bev, corners_lidar, obb_axes, obb_dims, 'orangered')

if var2_obb_ok:
    _bev_foot(ax_bev, corners_lidar_v2,
              'limegreen', '-.',  2.5,
              f'VAR2  {obb_dims_v2[0]:.2f}x{obb_dims_v2[1]:.2f}x{obb_dims_v2[2]:.2f} m')
    _bev_fwd(ax_bev, corners_lidar_v2, obb_axes_v2, obb_dims_v2, 'limegreen')

if var3_obb_ok:
    _bev_foot(ax_bev, corners_lidar_v3,
              'mediumpurple', (0, (3,1,1,1)), 2.5,
              f'VAR3  {obb_dims_v3[0]:.2f}x{obb_dims_v3[1]:.2f}x{obb_dims_v3[2]:.2f} m')
    _bev_fwd(ax_bev, corners_lidar_v3, obb_axes_v3, obb_dims_v3, 'mediumpurple')

ax_bev.set_xlim(ctr_cmp[0] - r_cmp, ctr_cmp[0] + r_cmp)
ax_bev.set_ylim(ctr_cmp[1] - r_cmp, ctr_cmp[1] + r_cmp)
ax_bev.set_aspect('equal', adjustable='box')
ax_bev.grid(alpha=0.35)
ax_bev.set_xlabel('LiDAR X (m)'); ax_bev.set_ylabel('LiDAR Y (m)')
ax_bev.set_title(f'BEV — Scene {NUSCENES_SCENE_IDX} / Frame {FRAME_IDX}\n'
                 f'ground_z={ground_z:+.3f}m ({_gnd_src})  '
                 f'blue=Global  orange=Local  green=VAR2'
                 + ('  purple=VAR3' if var3_obb_ok else ''))
ax_bev.legend(loc='upper right', fontsize=8, facecolor='k', labelcolor='w', framealpha=0.7)
plt.tight_layout()
plt.show()
